In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import hashlib
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)


In [3]:
# =========================================================
# Path Setup
# =========================================================
CATEGORY_ID = "face"
CATEGORY_FOLDER = "facial_skincare"
CATEGORY_LABEL = "Facial Skincare"
STAGE = "stage0_user_regime_sampling"
SAMPLING_ARTIFACT_SCHEMA_VERSION = "user_regime_sampling_v1"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories") / CATEGORY_FOLDER

os.chdir(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_ITEMS_DIR = PROJECT_ROOT / "data" / "processed" / "items"
SAMPLING_DIR = PROJECT_ROOT / "data" / "processed" / "user_sampling"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / STAGE

REVIEWS_PATH = PROJECT_ROOT / "data" / "raw" / "reviews_Skin_Care_Face_W2_2019_2023.parquet"
ITEM_SCHEMA_PATH = PROCESSED_ITEMS_DIR / "face_item_schema.parquet"
SAMPLED_USERS_PARQUET_PATH = SAMPLING_DIR / "face_user_regime_sample.parquet"
SAMPLED_USERS_COMPAT_PARQUET_PATH = SAMPLING_DIR / "sampled_users_by_regime.parquet"
SAMPLED_USERS_CSV_PATH = SAMPLING_DIR / "face_user_regime_sample.csv"
SAMPLED_USER_REJECTIONS_PARQUET_PATH = SAMPLING_DIR / "face_user_regime_rejections.parquet"
SAMPLED_ITEMS_PARQUET_PATH = SAMPLING_DIR / "face_sampled_experiment_items.parquet"
SAMPLED_ITEMS_COMPAT_PARQUET_PATH = SAMPLING_DIR / "sampled_experiment_items.parquet"
HELDOUT_REVIEW_POOL_PATH = SAMPLING_DIR / "face_heldout_review_pool.parquet"
FINAL_SAMPLING_POOL_PATH = SAMPLING_DIR / "face_final_sampling_pool.parquet"
USER_PRIOR_REVIEW_HISTORY_PATH = SAMPLING_DIR / "face_user_prior_review_history.parquet"
TRAIN_PRIOR_REVIEW_HISTORY_PATH = SAMPLING_DIR / "face_user_prior_review_history_training.parquet"
SAMPLING_MANIFEST_PATH = SAMPLING_DIR / "face_user_regime_sampling_manifest.json"


EVALUATION_WINDOW_MONTHS = 9
EVALUATION_WINDOW_END = pd.Timestamp("2023-09-12T14:52:26.427000+00:00")
EVALUATION_WINDOW_START = EVALUATION_WINDOW_END - pd.DateOffset(months=EVALUATION_WINDOW_MONTHS)
EVALUATION_WINDOW_START_INCLUSIVE = False
# The frozen historical-review cutoff is the exclusive start of the evaluation window.
TRAIN_REVIEW_CUTOFF_EXCLUSIVE = EVALUATION_WINDOW_START
if TRAIN_REVIEW_CUTOFF_EXCLUSIVE != EVALUATION_WINDOW_START:
    raise RuntimeError(
        "The historical-review cutoff must equal the evaluation-window start."
    )

MAX_TARGETS_PER_USER = 1
TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
TARGET_PRIOR_SAME_ITEM_POLICY = "exclude_target_candidates_with_prior_same_parent_asin"
MAX_TARGET_RANK_ALLOWED = 5
SUPPORTED_TARGET_SELECTION_MODES = {"strict_last_review", "most_recent_eligible_review", "recent_eligible_review_rank_le5"}
if TARGET_SELECTION_MODE not in SUPPORTED_TARGET_SELECTION_MODES:
    raise RuntimeError(f"Unsupported TARGET_SELECTION_MODE: {TARGET_SELECTION_MODE}")

SAMPLING_DECISION_SOURCE = "embedded_final_sampling_decision_from_prior_eligibility_audits"
FINAL_WINDOW_CHOICE = "9_month"

REGIME_ORDER = ["cold", "weak", "moderate", "strong"]

REQUESTED_SAMPLE_N_BY_REGIME = {
    regime: "match_minimal_training_prior_supply"
    for regime in REGIME_ORDER
}
TARGET_PER_REGIME = None
SAMPLE_N_BY_REGIME = {}
USER_REGIME_QUOTAS = {}
EXPECTED_FINAL_TOTAL_N = None
EXPECTED_UNIQUE_USERS_N = None

TARGET_TOTAL_N = None
TARGET_UNIQUE_USERS = None

# Dynamic benchmark contract: Notebook 03 derives the balanced source-pool size
# from the minimal strict training-period prior-history supply across regimes.
EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS = None
EXPECTED_ELIGIBLE_POOL_TOTAL_N = None
DOWNSTREAM_QUERY_BALANCE_N_PER_REGIME = None
DOWNSTREAM_BALANCED_QUERY_REGIME_COUNTS = None
DOWNSTREAM_BALANCED_QUERY_TOTAL_N = None
DOWNSTREAM_QUERY_BALANCING_BASIS = "minimal_training_prior_valid_regime_supply"

REGIME_DEFINITION = {
    "cold": "prior_review_n == 0",
    "weak": "1 <= prior_review_n <= 4",
    "moderate": "5 <= prior_review_n <= 9",
    "strong": "prior_review_n >= 10",
}

MIN_TARGET_REVIEW_TOKENS = 8
MIN_QUERY_SAFE_TOKEN_COUNT = 4
MIN_QUERY_SAFE_SIGNAL_FAMILIES = 2
MIN_QUERY_SAFE_SIGNAL_TOTAL = 2

FINAL_MIN_QUERY_TOKENS = 5
FINAL_MAX_QUERY_TOKENS = 18
FINAL_MIN_QUERY_SAFE_SIGNAL_FAMILIES = 2
FINAL_MIN_QUERY_SAFE_SIGNAL_TOTAL = 3

SAMPLING_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Evaluation window:", EVALUATION_WINDOW_START, "< review_datetime <=", EVALUATION_WINDOW_END)
print("Target sample: dynamic, balanced to minimal-regime strict training-prior supply")
print("Target selection mode:", TARGET_SELECTION_MODE)
print("Max target rank allowed:", MAX_TARGET_RANK_ALLOWED)


Project root: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare
Evaluation window: 2022-12-12 14:52:26.427000+00:00 < review_datetime <= 2023-09-12 14:52:26.427000+00:00
Target sample: dynamic, balanced to minimal-regime strict training-prior supply
Target selection mode: recent_eligible_review_rank_le5
Max target rank allowed: 5


### Common framework alignment note

This notebook applies the category-common sampling contract. It selects one held-out target review per eligible user from the user’s five most recent reviews in the evaluation window. All histories are recomputed relative to the selected target, restricted to interactions strictly earlier than that target, and exclude the target parent item. Stage 1 histories are additionally restricted to the frozen training cutoff. Regime labels are assigned from the resulting effective pre-target history.


In [4]:
# =========================================================
# Common User Sampling Framework Contract
# =========================================================
COMMON_USER_SAMPLING_FRAMEWORK_VERSION = "common_user_sampling_framework_v1"

COMMON_SAMPLING_COLUMNS = {
    "case_id": "case_id",
    "user_id": "user_id",
    "target_item_id": "parent_asin",
    "target_timestamp_ms": "target_timestamp_ms",
    "target_review_datetime": "review_datetime",
    "regime": "regime",
    "prior_count": "prior_review_n",
    "target_rank": "target_rank_desc",
    "target_selection_mode": "target_selection_mode",
    "n_pre_target_interactions": "n_pre_target_interactions",
    "n_pre_target_unique_items": "n_pre_target_unique_items",
    "n_train_safe_interactions": "n_train_safe_interactions",
    "n_train_safe_unique_items": "n_train_safe_unique_items",
    "stage1_profile_available": "stage1_profile_available",
    "stage2_profile_available": "stage2_profile_available",
}

COMMON_PRIOR_HISTORY_COLUMNS = {
    "case_id": "case_id",
    "user_id": "user_id",
    "target_item_id": "parent_asin",
    "target_timestamp_ms": "target_timestamp_ms",
    "prior_item_id": "prior_item_id",
    "prior_timestamp_ms": "prior_timestamp_ms",
}

COMMON_USER_SAMPLING_OUTPUT_CONTRACT = {
    "category_id": CATEGORY_ID,
    "artifact_schema_version": SAMPLING_ARTIFACT_SCHEMA_VERSION,
    "framework_version": COMMON_USER_SAMPLING_FRAMEWORK_VERSION,
    "regime_order": REGIME_ORDER,
    "regime_definition": REGIME_DEFINITION,
    "category_specific_regime_design": bool(globals().get("CATEGORY_SPECIFIC_REGIME_DESIGN", False)),
    "moderate_regime_used": bool("moderate" in REGIME_ORDER),
    "sampling_columns": COMMON_SAMPLING_COLUMNS,
    "prior_history_columns": COMMON_PRIOR_HISTORY_COLUMNS,
    "policy_note": "Sample-size quotas, target windows, rank limits, and eligibility thresholds remain category-specific; only the structural output contract is shared.",
}

COMMON_USER_SAMPLING_CONTRACT_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_common_user_sampling_contract.json"
COMMON_USER_SAMPLING_COLUMNS_PATH = OUTPUT_DIR / f"{CATEGORY_ID}_common_user_sampling_columns.csv"

# Shared critical sampling contract. This block is identical across categories.
COMMON_CRITICAL_SAMPLING_CONTRACT = {
    "version": "fresh_unified_v2",
    "candidate_history": "recomputed_for_each_target_candidate",
    "strict_history_predicate": "prior_timestamp_ms < target_timestamp_ms",
    "training_history_predicate": "prior_timestamp_ms < min(target_timestamp_ms, training_cutoff_ms)",
    "target_item_policy": "exclude_target_parent_from_both_histories_and_reject_previously_reviewed_target",
    "regime_source": "n_pre_target_interactions",
    "target_selection": "most_recent_eligible_target_within_five_most_recent_window_reviews",
    "one_target_per_user": True,
    "non_cold_stage1_requirement": "at_least_one_training_safe_prior_item",
}
COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256 = hashlib.sha256(
    json.dumps(
        COMMON_CRITICAL_SAMPLING_CONTRACT,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()
EXPECTED_COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256 = "13d580fcdc9e2d5bf25c48090d05681ed6b7b2b894a42421541c75763a342259"
if COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256 != EXPECTED_COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256:
    raise RuntimeError("Common critical sampling contract hash mismatch.")


### Sampling design note

The 698-case-per-regime sample created in this notebook is a pre-query source sample, not the final analytical benchmark. Downstream query-sufficiency and eligibility checks reduce the final Skincare benchmark to 572 cases per regime, or 2,288 cases in total. Replacements are deterministic, restricted to the same regime, and do not increase the final quota.


In [5]:
# =========================================================
# Helper Functions
# =========================================================
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()

def tokenize_text(text):
    return re.findall(r"[A-Za-z0-9']+", normalize_space(text).lower())

def is_nonempty_value(value):
    if value is None:
        return False
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return any(is_nonempty_value(v) for v in value)
    try:
        if pd.isna(value):
            return False
    except TypeError:
        pass
    return normalize_space(value).lower() not in {"", "none", "null", "nan", "n/a", "na"}

def parquet_columns(path: Path):
    try:
        import pyarrow.parquet as pq
        return list(pq.ParquetFile(path).schema.names)
    except Exception:
        return list(pd.read_parquet(path).columns)

def first_existing_column(columns, candidates, required=True, label="column"):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    if required:
        raise RuntimeError(f"Missing required {label}. Tried {candidates}.")
    return None

def to_review_timestamp_ms(series: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(series):
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.dropna().empty:
        parsed = pd.to_datetime(series, utc=True, errors="coerce")
        return (parsed.astype("int64") // 1_000_000).where(parsed.notna(), np.nan)
    if numeric.dropna().max() < 10**11:
        numeric = numeric * 1000
    return numeric

def normalize_bool_flag(value):
    if value is None or pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer, float, np.floating)):
        return bool(value)
    return normalize_space(value).lower() in {"true", "yes", "y", "1", "discontinued"}

def regime_from_prior_count(prior_n):
    n = int(0 if pd.isna(prior_n) else prior_n)
    if n == 0:
        return "cold"
    if 1 <= n <= 4:
        return "weak"
    if 5 <= n <= 9:
        return "moderate"
    if n >= 10:
        return "strong"
    return "other"

def sampling_bracket_from_prior_count(prior_n):
    n = int(0 if pd.isna(prior_n) else prior_n)
    regime = regime_from_prior_count(n)
    if regime == "cold":
        return "0"
    if regime in {"weak", "moderate"}:
        return str(n)
    if regime == "strong":
        return "10+"
    return ""

def make_case_id(user_id, parent_asin, timestamp_ms):
    raw = f"{user_id}|{parent_asin}|{int(timestamp_ms)}"
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:12]

def sort_by_regime_bracket(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "regime" not in df.columns:
        return df.reset_index(drop=True)
    regime_order = {"cold": 0, "weak": 1, "moderate": 2, "strong": 3}
    out = df.copy()
    out["_regime_order"] = out["regime"].map(lambda x: regime_order.get(str(x), 999)).astype(int)
    out["_bracket_order"] = out.get("sampling_bracket", pd.Series("", index=out.index)).map(lambda x: 10 if str(x) == "10+" else int(x) if str(x).isdigit() else 999)
    return out.sort_values(["_regime_order", "_bracket_order"]).drop(columns=["_regime_order", "_bracket_order"]).reset_index(drop=True)


In [6]:
# =========================================================
# Item Schema Loading
# =========================================================
FORBIDDEN_REVIEW_EXACT_COLS = {
    "target_review_text", "heldout_review_text", "review_text", "review_text_agg_train",
    "concern_review_raw", "concern_review_raw_text", "concern_review_evidence_text", "has_review_evidence",
}
FORBIDDEN_REVIEW_PREFIXES = ("review_text_", "target_review_", "concern_review_", "review_evidence_", "review_concern_")
SAFE_REVIEW_REPUTATION_PREFIXES = ("review_reputation_", "historical_review_")

def forbidden_review_columns(columns):
    forbidden = []
    for col in columns:
        col_str = str(col)
        if col_str in FORBIDDEN_REVIEW_EXACT_COLS or col_str.startswith(FORBIDDEN_REVIEW_PREFIXES):
            forbidden.append(col_str)
    return sorted(set(forbidden))

item_schema = pd.read_parquet(ITEM_SCHEMA_PATH)
if "parent_asin" not in item_schema.columns:
    raise RuntimeError("Item schema must contain parent_asin.")
item_forbidden_cols = forbidden_review_columns(item_schema.columns)
if item_forbidden_cols:
    raise RuntimeError(f"Unsafe target-review/raw-review item schema columns found: {item_forbidden_cols}")

item_schema = item_schema[item_schema["parent_asin"].notna()].copy()
item_schema["parent_asin"] = item_schema["parent_asin"].astype(str).str.strip()
item_schema = item_schema[item_schema["parent_asin"].str.len() > 0].copy()
item_schema = item_schema.sort_values("parent_asin", kind="mergesort").drop_duplicates("parent_asin", keep="first").reset_index(drop=True)

title_col_for_sampling = "title" if "title" in item_schema.columns else None
brand_col_for_sampling = "brand_meta" if "brand_meta" in item_schema.columns else ("brand" if "brand" in item_schema.columns else None)
discontinued_col_for_sampling = "is_discontinued" if "is_discontinued" in item_schema.columns else ("discontinued" if "discontinued" in item_schema.columns else None)

item_schema["item_is_discontinued"] = item_schema[discontinued_col_for_sampling].map(normalize_bool_flag) if discontinued_col_for_sampling else False
item_schema["item_metadata_exists"] = True

item_context_cols = [
    "parent_asin", "item_metadata_exists", "item_is_discontinued",
    "title", "brand_meta", "brand", "manufacturer", "seller",
    "sub_category_norm_text", "form_norm_text", "product_type_norm_text", "texture_norm_text",
    "concern_claim_norm_text", "benefit_norm_text", "ingredient_norm_text", "skin_type_norm_text",
    "review_reputation_available_flag", "historical_review_reputation_text",
]
item_context_cols = [col for col in item_context_cols if col in item_schema.columns]
item_context_df = item_schema[item_context_cols].copy()

print("Item schema rows:", len(item_schema))
print("Safe review reputation columns present:", sorted([c for c in item_schema.columns if str(c).startswith(SAFE_REVIEW_REPUTATION_PREFIXES)])[:20])


Item schema rows: 78196
Safe review reputation columns present: ['historical_review_count_pre_window', 'historical_review_reputation_enabled', 'historical_review_reputation_quarantined', 'historical_review_reputation_text', 'review_reputation_available_flag', 'review_reputation_benefit_text', 'review_reputation_concern_text', 'review_reputation_cutoff_exclusive', 'review_reputation_facet_text', 'review_reputation_ingredient_text', 'review_reputation_only_text', 'review_reputation_product_type_or_form_texture_text', 'review_reputation_signal_family_count', 'review_reputation_signal_total_count', 'review_reputation_skin_type_text', 'review_reputation_source']


In [7]:
# =========================================================
# Query Eligibility
# =========================================================
ASIN_PATTERN = re.compile(r"\bb0[a-z0-9]{8}\b", re.IGNORECASE)
PACKAGE_IDENTIFIER_PATTERN = re.compile(
    r"\b(\d+(?:\.\d+)?\s?(?:oz|fl\.?\s?oz|ml|g|gram|grams|ct|count|pack|packs|pcs|piece|pieces|%|percent|mg|mcg|iu)|spf\s?\d+|asin|seller|manufacturer|barcode|upc)\b",
    re.IGNORECASE,
)
DIRECT_CUE_STOPWORDS = {
    "the", "and", "or", "for", "with", "from", "skin", "care", "face", "facial", "product",
    "cream", "serum", "cleanser", "moisturizer", "lotion", "gel", "mask", "toner", "oil", "balm", "spf", "set",
}
SIGNAL_PATTERNS = {
    "concern": {
        "acne": [r"\bacne\b", r"\bbreakouts?\b", r"\bblemishes?\b", r"\bpimples?\b"],
        "redness": [r"\bredness\b", r"\bred skin\b"],
        "dark spots": [r"\bdark spots?\b", r"\bhyperpigmentation\b", r"\buneven tone\b"],
        "pores": [r"\bpores?\b", r"\blarge pores?\b"],
        "dryness": [r"\bdry(ness)?\b", r"\bdehydrat(ed|ion)\b", r"\bflaky\b"],
        "oiliness": [r"\boily\b", r"\bgreasy\b", r"\bshine\b"],
        "dullness": [r"\bdull(ness)?\b"],
        "fine lines": [r"\bfine lines?\b", r"\bwrinkles?\b"],
        "texture": [r"\btexture\b", r"\brough\b", r"\bbumpy\b"],
    },
    "skin_type": {
        "dry skin": [r"\bdry skin\b"],
        "oily skin": [r"\boily skin\b"],
        "combination skin": [r"\bcombination skin\b", r"\bcombo skin\b"],
        "sensitive skin": [r"\bsensitive skin\b", r"\bsensitiv(e|ity)\b"],
        "acne-prone skin": [r"\bacne prone\b", r"\bacne-prone\b"],
    },
    "benefit": {
        "hydrating": [r"\bhydrat\w*", r"\bmoisturi[sz]\w*"],
        "soothing": [r"\bsooth\w*", r"\bcalm\w*"],
        "brightening": [r"\bbrighten\w*", r"\bglow\w*"],
        "smoothing": [r"\bsmooth\w*", r"\bsoften\w*"],
        "firming": [r"\bfirm\w*", r"\btighten\w*"],
        "cleansing": [r"\bcleanse\w*", r"\bremoves? makeup\b"],
        "exfoliating": [r"\bexfoliat\w*", r"\bpeel\b"],
    },
    "ingredient": {
        "hyaluronic acid": [r"\bhyaluronic acid\b"],
        "niacinamide": [r"\bniacinamide\b"],
        "retinol": [r"\bretinol\b", r"\bretinoid\b"],
        "vitamin c": [r"\bvitamin c\b"],
        "ceramide": [r"\bceramides?\b"],
        "salicylic acid": [r"\bsalicylic acid\b", r"\bbha\b"],
        "glycolic acid": [r"\bglycolic acid\b", r"\baha\b"],
        "centella": [r"\bcentella\b", r"\bcica\b"],
        "aloe": [r"\baloe\b"],
        "tea tree": [r"\btea tree\b"],
    },
    "product_type_or_form_texture": {
        "cleanser": [r"\bcleanser\b", r"\bface wash\b"],
        "serum": [r"\bserum\b"],
        "moisturizer": [r"\bmoisturi[sz]er\b", r"\bcream\b", r"\blotion\b"],
        "toner": [r"\btoner\b"],
        "mask": [r"\bmask\b"],
        "sunscreen": [r"\bsunscreen\b", r"\bspf\b"],
        "balm": [r"\bbalm\b"],
        "oil": [r"\boil\b"],
        "gel": [r"\bgel\b"],
        "foam": [r"\bfoam\w*\b"],
        "lightweight": [r"\blight ?weight\b"],
        "rich texture": [r"\brich\b", r"\bthick\b"],
        "non-greasy": [r"\bnon greasy\b", r"\bnon-greasy\b"],
    },
}

def cue_terms_from_value(value):
    text = normalize_space(value).lower()
    if not text:
        return []
    terms = [text]
    terms.extend([tok for tok in tokenize_text(text) if len(tok) >= 4 and tok not in DIRECT_CUE_STOPWORDS])
    return list(dict.fromkeys([term for term in terms if term]))

def remove_direct_item_cues(text, row):
    scrubbed = normalize_space(text).lower()
    cue_terms = []
    for col in ["title", brand_col_for_sampling, "brand_meta", "brand", "manufacturer", "seller", "parent_asin"]:
        if col and col in row.index:
            cue_terms.extend(cue_terms_from_value(row.get(col, "")))
    for term in sorted(set(cue_terms), key=len, reverse=True):
        if len(term) >= 4:
            scrubbed = re.sub(r"(?<![a-z0-9])" + re.escape(term) + r"(?![a-z0-9])", " ", scrubbed)
    scrubbed = ASIN_PATTERN.sub(" ", scrubbed)
    scrubbed = PACKAGE_IDENTIFIER_PATTERN.sub(" ", scrubbed)
    return normalize_space(scrubbed)

def extract_signal_counts(query_safe_text):
    lowered = normalize_space(query_safe_text).lower()
    family_hits = {}
    signal_total = 0
    for family, labels in SIGNAL_PATTERNS.items():
        label_hits = 0
        for patterns in labels.values():
            if any(re.search(pattern, lowered) for pattern in patterns):
                label_hits += 1
        family_hits[family] = int(label_hits > 0)
        signal_total += int(label_hits)
    return int(sum(family_hits.values())), int(signal_total), family_hits

def query_convertibility_diagnostics(row):
    review_text = normalize_space(row.get("target_review_text", ""))
    target_review_token_count = len(tokenize_text(review_text))
    query_safe_text = remove_direct_item_cues(review_text, row)
    query_safe_token_count = len(tokenize_text(query_safe_text))
    family_count, signal_total_count, family_hits = extract_signal_counts(query_safe_text)
    item_metadata_exists = bool(row.get("item_metadata_exists", False))
    item_discontinued = bool(row.get("item_is_discontinued", False))
    if review_text == "":
        reason = "missing_review_text"
    elif target_review_token_count < MIN_TARGET_REVIEW_TOKENS:
        reason = "short_review"
    elif not item_metadata_exists:
        reason = "missing_item_metadata"
    elif item_discontinued:
        reason = "discontinued_item"
    elif query_safe_token_count < MIN_QUERY_SAFE_TOKEN_COUNT:
        reason = "low_query_safe_token_count"
    elif family_count < MIN_QUERY_SAFE_SIGNAL_FAMILIES:
        reason = "low_signal_family_count"
    elif signal_total_count < MIN_QUERY_SAFE_SIGNAL_TOTAL:
        reason = "low_signal_total_count"
    else:
        reason = "eligible"
    out = {
        "target_review_token_count": int(target_review_token_count),
        "query_safe_token_count": int(query_safe_token_count),
        "query_safe_signal_family_count": int(family_count),
        "query_safe_signal_total_count": int(signal_total_count),
        "query_convertible_flag": int(reason == "eligible"),
        "query_convertibility_failure_reason": reason,
    }
    out.update({f"query_safe_has_{family}": int(value) for family, value in family_hits.items()})
    return pd.Series(out)


In [8]:
# =========================================================
# Review Loading
# =========================================================
# Raw reviews are used only for fixed-window user-level target sampling, target review text
review_columns = parquet_columns(REVIEWS_PATH)
if "parent_asin" not in review_columns:
    raise RuntimeError("Raw reviews must contain parent_asin. Do not fallback from asin.")

review_user_col = first_existing_column(review_columns, ["user_id", "reviewerID", "reviewer_id", "customer_id"], label="review user id")
review_ts_col = first_existing_column(review_columns, ["timestamp_ms", "timestamp", "unixReviewTime", "review_time"], label="review timestamp")
review_text_col = first_existing_column(review_columns, ["text", "body", "reviewText", "review_text", "content"], required=False, label="review text")
review_title_col = first_existing_column(review_columns, ["title", "review_title", "summary"], required=False, label="review title")
if review_text_col is None and review_title_col is None:
    raise RuntimeError("Raw reviews must contain text or title.")

review_load_cols = ["parent_asin", review_user_col, review_ts_col]
for optional_col in [review_title_col, review_text_col]:
    if optional_col and optional_col not in review_load_cols:
        review_load_cols.append(optional_col)
raw_reviews = pd.read_parquet(REVIEWS_PATH, columns=review_load_cols)
print("Raw review rows loaded:", len(raw_reviews))
print("Loaded review columns:", review_load_cols)

reviews = pd.DataFrame({
    "user_id": raw_reviews[review_user_col].fillna("").astype(str),
    "parent_asin": raw_reviews["parent_asin"].fillna("").astype(str),
    "review_timestamp_ms": to_review_timestamp_ms(raw_reviews[review_ts_col]),
})
reviews["review_datetime"] = pd.to_datetime(reviews["review_timestamp_ms"], unit="ms", utc=True, errors="coerce")
text_series = raw_reviews[review_text_col].map(normalize_space) if review_text_col else pd.Series("", index=raw_reviews.index)
title_series = raw_reviews[review_title_col].map(normalize_space) if review_title_col else pd.Series("", index=raw_reviews.index)
if review_text_col and review_title_col:
    reviews["target_review_text"] = (title_series + " " + text_series).map(normalize_space)
elif review_text_col:
    reviews["target_review_text"] = text_series.map(normalize_space)
else:
    reviews["target_review_text"] = title_series.map(normalize_space)

reviews = reviews.dropna(subset=["review_timestamp_ms", "review_datetime"]).copy()
reviews["review_timestamp_ms"] = reviews["review_timestamp_ms"].astype("int64")
reviews = reviews[(reviews["user_id"].str.len() > 0) & (reviews["parent_asin"].str.len() > 0)].copy()

print("Standardized review rows:", len(reviews))
print("Min/max review datetime:", reviews["review_datetime"].min(), reviews["review_datetime"].max())


Raw review rows loaded: 990619
Loaded review columns: ['parent_asin', 'user_id', 'timestamp', 'title', 'text']
Standardized review rows: 990619
Min/max review datetime: 2019-01-01 00:01:33.712000+00:00 2023-09-12 14:52:26.427000+00:00


In [9]:
# =========================================================
# Target Candidate Selection
# =========================================================
evaluation_reviews = reviews[
    (reviews["review_datetime"] > EVALUATION_WINDOW_START) &
    (reviews["review_datetime"] <= EVALUATION_WINDOW_END)
].copy()
if evaluation_reviews.empty:
    raise RuntimeError("No reviews found in the configured evaluation window.")

evaluation_reviews = evaluation_reviews.sort_values(["user_id", "review_timestamp_ms", "parent_asin"], ascending=[True, False, True]).reset_index(drop=True)
evaluation_reviews["target_rank_desc"] = evaluation_reviews.groupby("user_id").cumcount() + 1
candidate_review_counts_by_user = evaluation_reviews.groupby("user_id").size().reset_index(name="candidate_review_n")
users_with_multiple_candidate_reviews = int((candidate_review_counts_by_user["candidate_review_n"] > 1).sum())
avg_candidate_reviews_per_user = float(candidate_review_counts_by_user["candidate_review_n"].mean())
max_candidate_reviews_per_user = int(candidate_review_counts_by_user["candidate_review_n"].max())

target_candidates = evaluation_reviews.copy().reset_index(drop=True)
if MAX_TARGET_RANK_ALLOWED is not None:
    target_candidates = target_candidates[
        target_candidates["target_rank_desc"].le(MAX_TARGET_RANK_ALLOWED)
    ].copy().reset_index(drop=True)
if MAX_TARGET_RANK_ALLOWED is not None and not target_candidates["target_rank_desc"].le(MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"target_rank_desc must be <= {MAX_TARGET_RANK_ALLOWED}.")
target_candidates["target_candidate_row_id"] = np.arange(len(target_candidates), dtype=np.int64)
target_candidates["case_id"] = [
    make_case_id(user_id, parent_asin, ts)
    for user_id, parent_asin, ts in zip(target_candidates["user_id"], target_candidates["parent_asin"], target_candidates["review_timestamp_ms"])
]
target_candidates = target_candidates.merge(item_context_df, on="parent_asin", how="left")
target_candidates["item_metadata_exists"] = target_candidates["item_metadata_exists"].fillna(False).astype(bool)
target_candidates["item_is_discontinued"] = target_candidates["item_is_discontinued"].fillna(False).astype(bool)

print("Evaluation-window review candidate rows:", len(evaluation_reviews))
print(f"Rank <= {MAX_TARGET_RANK_ALLOWED} target candidates:", len(target_candidates))
print("Unique users in fixed evaluation window:", target_candidates["user_id"].nunique())
print("Users with multiple candidate reviews:", users_with_multiple_candidate_reviews)
print("Average candidate reviews per user:", round(avg_candidate_reviews_per_user, 4))
print("Max candidate reviews per user:", max_candidate_reviews_per_user)


Evaluation-window review candidate rows: 121064
Rank <= 5 target candidates: 111036
Unique users in fixed evaluation window: 92412
Users with multiple candidate reviews: 11280
Average candidate reviews per user: 1.31
Max candidate reviews per user: 133


In [10]:
# =========================================================
# Candidate-Specific Prior History and Regimes
# =========================================================
def compute_prior_counts(all_reviews: pd.DataFrame, target_df: pd.DataFrame) -> pd.DataFrame:
    history = (
        all_reviews[["user_id", "parent_asin", "review_timestamp_ms"]]
        .dropna(subset=["review_timestamp_ms"])
        .copy()
    )
    history = history.sort_values(["user_id", "review_timestamp_ms", "parent_asin"])
    history_by_user = {
        user_id: group.copy()
        for user_id, group in history.groupby("user_id", sort=False)
    }

    rows = []
    required_target_cols = [
        "target_candidate_row_id",
        "case_id",
        "user_id",
        "parent_asin",
        "review_timestamp_ms",
    ]
    missing_target_cols = [col for col in required_target_cols if col not in target_df.columns]
    if missing_target_cols:
        raise RuntimeError(
            f"Target candidate dataframe is missing required prior-count keys: {missing_target_cols}"
        )
    if target_df["target_candidate_row_id"].duplicated().any():
        raise RuntimeError("target_candidate_row_id must be unique before prior-count computation.")

    training_cutoff_ms = int(TRAIN_REVIEW_CUTOFF_EXCLUSIVE.value // 1_000_000)

    for row in target_df[required_target_cols].itertuples(index=False):
        user_history = history_by_user.get(row.user_id)
        target_timestamp_ms = int(row.review_timestamp_ms)
        target_item_id = str(row.parent_asin)

        if user_history is None or user_history.empty:
            pre_target_history = history.iloc[0:0]
            prior_same_item = history.iloc[0:0]
        else:
            strictly_earlier = user_history[
                user_history["review_timestamp_ms"] < target_timestamp_ms
            ]
            prior_same_item = strictly_earlier[
                strictly_earlier["parent_asin"].astype(str).eq(target_item_id)
            ]
            pre_target_history = strictly_earlier[
                ~strictly_earlier["parent_asin"].astype(str).eq(target_item_id)
            ]

        effective_training_cutoff_ms = min(target_timestamp_ms, training_cutoff_ms)
        train_safe_history = pre_target_history[
            pre_target_history["review_timestamp_ms"] < effective_training_cutoff_ms
        ]

        n_pre_target_interactions = int(len(pre_target_history))
        n_pre_target_unique_items = int(pre_target_history["parent_asin"].nunique())
        n_train_safe_interactions = int(len(train_safe_history))
        n_train_safe_unique_items = int(train_safe_history["parent_asin"].nunique())

        rows.append({
            "target_candidate_row_id": int(row.target_candidate_row_id),
            "case_id": row.case_id,
            "n_pre_target_interactions": n_pre_target_interactions,
            "n_pre_target_unique_items": n_pre_target_unique_items,
            "n_train_safe_interactions": n_train_safe_interactions,
            "n_train_safe_unique_items": n_train_safe_unique_items,
            "stage1_profile_available": bool(n_train_safe_unique_items >= 1),
            "stage2_profile_available": bool(n_pre_target_unique_items >= 1),
            "prior_same_item_review_n": int(len(prior_same_item)),
        })

    return pd.DataFrame(rows)


prior_counts_df = compute_prior_counts(reviews, target_candidates)
heldout_review_pool_df = target_candidates.merge(
    prior_counts_df,
    on=["target_candidate_row_id", "case_id"],
    how="left",
    validate="one_to_one",
)
if len(heldout_review_pool_df) != len(target_candidates):
    raise RuntimeError("Prior-count merge changed the number of target candidate rows.")
if heldout_review_pool_df["target_candidate_row_id"].duplicated().any():
    raise RuntimeError("Prior-count merge produced duplicate target_candidate_row_id values.")

count_columns = [
    "n_pre_target_interactions",
    "n_pre_target_unique_items",
    "n_train_safe_interactions",
    "n_train_safe_unique_items",
    "prior_same_item_review_n",
]
heldout_review_pool_df[count_columns] = (
    heldout_review_pool_df[count_columns].fillna(0).astype(int)
)
heldout_review_pool_df["stage1_profile_available"] = (
    heldout_review_pool_df["stage1_profile_available"].fillna(False).astype(bool)
)
heldout_review_pool_df["stage2_profile_available"] = (
    heldout_review_pool_df["stage2_profile_available"].fillna(False).astype(bool)
)

# Compatibility aliases retain strict pre-target history, not training-cutoff history.
heldout_review_pool_df["prior_review_n"] = heldout_review_pool_df[
    "n_pre_target_interactions"
].astype(int)
heldout_review_pool_df["prior_item_n"] = heldout_review_pool_df[
    "n_pre_target_unique_items"
].astype(int)
heldout_review_pool_df["strict_training_prior_rows"] = heldout_review_pool_df[
    "n_train_safe_interactions"
].astype(int)
heldout_review_pool_df["strict_training_prior_unique_items"] = heldout_review_pool_df[
    "n_train_safe_unique_items"
].astype(int)
heldout_review_pool_df["raw_pre_target_prior_review_n"] = heldout_review_pool_df[
    "n_pre_target_interactions"
].astype(int)
heldout_review_pool_df["raw_pre_target_prior_item_n"] = heldout_review_pool_df[
    "n_pre_target_unique_items"
].astype(int)

heldout_review_pool_df["target_item_has_prior_same_item_review"] = (
    heldout_review_pool_df["prior_same_item_review_n"].gt(0).astype(int)
)
heldout_review_pool_df["target_item_repeat_prior_flag"] = (
    heldout_review_pool_df["target_item_has_prior_same_item_review"].astype(int)
)

heldout_review_pool_df["regime"] = heldout_review_pool_df[
    "n_pre_target_interactions"
].map(regime_from_prior_count)
heldout_review_pool_df["sampling_bracket"] = heldout_review_pool_df[
    "n_pre_target_interactions"
].map(sampling_bracket_from_prior_count)
heldout_review_pool_df = heldout_review_pool_df[
    heldout_review_pool_df["regime"].isin(REGIME_ORDER)
].copy()

if (
    heldout_review_pool_df["n_train_safe_interactions"]
    > heldout_review_pool_df["n_pre_target_interactions"]
).any():
    raise RuntimeError("Training-safe history cannot exceed strict pre-target history.")
if (
    heldout_review_pool_df["n_train_safe_unique_items"]
    > heldout_review_pool_df["n_pre_target_unique_items"]
).any():
    raise RuntimeError("Training-safe unique items cannot exceed strict pre-target unique items.")
if not heldout_review_pool_df["stage1_profile_available"].eq(
    heldout_review_pool_df["n_train_safe_unique_items"].ge(1)
).all():
    raise RuntimeError("stage1_profile_available does not match training-safe unique-item supply.")
if not heldout_review_pool_df["stage2_profile_available"].eq(
    heldout_review_pool_df["n_pre_target_unique_items"].ge(1)
).all():
    raise RuntimeError("stage2_profile_available does not match strict pre-target unique-item supply.")

repeat_target_candidate_n = int(heldout_review_pool_df["target_item_repeat_prior_flag"].sum())
repeat_target_candidate_user_n = int(
    heldout_review_pool_df.loc[
        heldout_review_pool_df["target_item_repeat_prior_flag"].eq(1),
        "user_id",
    ].nunique()
)

print("Held-out pool rows:", len(heldout_review_pool_df))
print(heldout_review_pool_df["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).to_string())
print("Target candidates with prior same-item review:", repeat_target_candidate_n)
print("Users with prior same-item target candidates:", repeat_target_candidate_user_n)
print("Non-cold candidates without Stage 1 training-safe history:", int(
    (
        heldout_review_pool_df["regime"].ne("cold")
        & ~heldout_review_pool_df["stage1_profile_available"]
    ).sum()
))
print("Target prior same-item policy:", TARGET_PRIOR_SAME_ITEM_POLICY)

Held-out pool rows: 111036
regime
cold        80603
weak        22618
moderate     2989
strong       4826
Target candidates with prior same-item review: 214
Users with prior same-item target candidates: 204
Non-cold candidates without Stage 1 training-safe history: 10630
Target prior same-item policy: exclude_target_candidates_with_prior_same_parent_asin


In [11]:
# =========================================================
# Target Selection
# =========================================================
query_diag_df = heldout_review_pool_df.apply(query_convertibility_diagnostics, axis=1)
heldout_review_pool_df = pd.concat(
    [heldout_review_pool_df.reset_index(drop=True), query_diag_df.reset_index(drop=True)],
    axis=1,
)

heldout_review_pool_df["target_review_token_len"] = heldout_review_pool_df["target_review_token_count"]
heldout_review_pool_df["review_richness_score"] = heldout_review_pool_df["query_safe_signal_total_count"]
heldout_review_pool_df["target_review_structure_signal_count"] = heldout_review_pool_df["query_safe_signal_family_count"]

heldout_review_pool_df["stage1_sampling_eligible"] = (
    heldout_review_pool_df["regime"].eq("cold")
    | heldout_review_pool_df["stage1_profile_available"]
)
heldout_review_pool_df["target_item_sampling_eligible"] = (
    heldout_review_pool_df["item_metadata_exists"]
    & ~heldout_review_pool_df["item_is_discontinued"]
    & heldout_review_pool_df["target_item_repeat_prior_flag"].eq(0)
    & heldout_review_pool_df["stage1_sampling_eligible"]
)

heldout_review_pool_df["sampling_rejection_reason"] = ""
heldout_review_pool_df.loc[
    heldout_review_pool_df["query_convertible_flag"].ne(1),
    "sampling_rejection_reason",
] = heldout_review_pool_df.loc[
    heldout_review_pool_df["query_convertible_flag"].ne(1),
    "query_convertibility_failure_reason",
].fillna("not_query_convertible")
heldout_review_pool_df.loc[
    heldout_review_pool_df["regime"].ne("cold")
    & ~heldout_review_pool_df["stage1_profile_available"],
    "sampling_rejection_reason",
] = "non_cold_without_training_safe_prior"
heldout_review_pool_df.loc[
    heldout_review_pool_df["item_metadata_exists"].ne(True),
    "sampling_rejection_reason",
] = "missing_item_metadata"
heldout_review_pool_df.loc[
    heldout_review_pool_df["item_is_discontinued"].eq(True),
    "sampling_rejection_reason",
] = "discontinued_item"
heldout_review_pool_df.loc[
    heldout_review_pool_df["target_item_repeat_prior_flag"].eq(1),
    "sampling_rejection_reason",
] = "target_item_previously_reviewed_by_same_user"

all_candidate_query_convertible_df = heldout_review_pool_df[
    heldout_review_pool_df["query_convertible_flag"].eq(1)
    & heldout_review_pool_df["target_item_sampling_eligible"].eq(True)
].copy()

top_rank_user_df = target_candidates[["user_id"]].drop_duplicates().copy()
eligible_top_rank_user_df = all_candidate_query_convertible_df[["user_id"]].drop_duplicates().copy()
users_with_no_eligible_recent_review_df = top_rank_user_df.merge(
    eligible_top_rank_user_df.assign(has_recent_eligible_review=True),
    on="user_id",
    how="left",
)
users_with_no_eligible_recent_review_df = users_with_no_eligible_recent_review_df[
    users_with_no_eligible_recent_review_df["has_recent_eligible_review"].ne(True)
].drop(columns=["has_recent_eligible_review"]).copy()

strict_last_user_pool_df = heldout_review_pool_df[
    heldout_review_pool_df["target_rank_desc"].eq(1)
].copy()
strict_last_query_convertible_pool_df = strict_last_user_pool_df[
    strict_last_user_pool_df["query_convertible_flag"].eq(1)
    & strict_last_user_pool_df["target_item_sampling_eligible"].eq(True)
].copy()
strict_last_query_convertible_pool_df["target_selection_mode"] = "strict_last_review"

most_recent_eligible_user_pool_df = (
    all_candidate_query_convertible_df
    .sort_values(
        ["user_id", "review_timestamp_ms", "parent_asin"],
        ascending=[True, False, True],
    )
    .drop_duplicates("user_id", keep="first")
    .copy()
)
most_recent_eligible_user_pool_df["target_selection_mode"] = "most_recent_eligible_review"

rank_limited_eligible_user_pool_df = (
    all_candidate_query_convertible_df
    .loc[lambda df: (
        pd.Series(True, index=df.index)
        if MAX_TARGET_RANK_ALLOWED is None
        else df["target_rank_desc"].le(MAX_TARGET_RANK_ALLOWED)
    )]
    .sort_values(
        ["user_id", "target_rank_desc", "review_timestamp_ms", "parent_asin"],
        ascending=[True, True, False, True],
        kind="mergesort",
    )
    .drop_duplicates("user_id", keep="first")
    .copy()
)
rank_limited_eligible_user_pool_df["target_selection_mode"] = "recent_eligible_review_rank_le5"

if TARGET_SELECTION_MODE == "strict_last_review":
    selected_user_level_pool_df = strict_last_query_convertible_pool_df.copy()
elif TARGET_SELECTION_MODE == "most_recent_eligible_review":
    selected_user_level_pool_df = most_recent_eligible_user_pool_df.copy()
elif TARGET_SELECTION_MODE == "recent_eligible_review_rank_le5":
    selected_user_level_pool_df = rank_limited_eligible_user_pool_df.copy()
else:
    raise RuntimeError(f"Unsupported TARGET_SELECTION_MODE: {TARGET_SELECTION_MODE}")

if selected_user_level_pool_df["user_id"].duplicated().any():
    raise RuntimeError("User-level selected pool contains duplicate user_id values.")
if selected_user_level_pool_df.groupby("user_id")["case_id"].nunique().max() > MAX_TARGETS_PER_USER:
    raise RuntimeError("User-level selected pool contains more than one target per user.")

review_case_supply_by_regime = (
    all_candidate_query_convertible_df["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)
strict_last_supply_by_regime = (
    strict_last_query_convertible_pool_df["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)
most_recent_eligible_supply_by_regime = (
    most_recent_eligible_user_pool_df["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)
rank_limited_eligible_supply_by_regime = (
    rank_limited_eligible_user_pool_df["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)

final_sampling_pool_df = selected_user_level_pool_df.reset_index(drop=True).copy()
if MAX_TARGET_RANK_ALLOWED is not None and not final_sampling_pool_df["target_rank_desc"].le(MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"Selected targets must have target_rank_desc <= {MAX_TARGET_RANK_ALLOWED}.")

selected_target_rank_counts = (
    final_sampling_pool_df["target_rank_desc"]
    .value_counts()
    .reindex(
        range(1, int(MAX_TARGET_RANK_ALLOWED) + 1),
        fill_value=0,
    )
    .astype(int)
    .to_dict()
)

if not final_sampling_pool_df["prior_review_n"].eq(
    final_sampling_pool_df["n_pre_target_interactions"]
).all():
    raise RuntimeError("prior_review_n must remain the strict pre-target interaction count.")
if not final_sampling_pool_df["prior_item_n"].eq(
    final_sampling_pool_df["n_pre_target_unique_items"]
).all():
    raise RuntimeError("prior_item_n must remain the strict pre-target unique-item count.")
if not final_sampling_pool_df["regime"].eq(
    final_sampling_pool_df["n_pre_target_interactions"].map(regime_from_prior_count)
).all():
    raise RuntimeError("Regime must be derived from strict pre-target history for the selected target.")

non_cold_zero_training_safe_prior_df = final_sampling_pool_df[
    final_sampling_pool_df["regime"].ne("cold")
    & final_sampling_pool_df["n_train_safe_unique_items"].lt(1)
].copy()
print("Non-cold selected cases with zero training-safe prior items:", len(non_cold_zero_training_safe_prior_df))
if len(non_cold_zero_training_safe_prior_df):
    display(non_cold_zero_training_safe_prior_df[[
        "case_id", "user_id", "parent_asin", "review_timestamp_ms", "regime",
        "n_pre_target_interactions", "n_train_safe_interactions",
        "n_train_safe_unique_items",
    ]].head(20))

bad_cold_history = final_sampling_pool_df[
    final_sampling_pool_df["regime"].eq("cold")
    & (
        final_sampling_pool_df["n_pre_target_interactions"].ne(0)
        | final_sampling_pool_df["n_train_safe_interactions"].ne(0)
    )
].copy()
if len(bad_cold_history):
    display(bad_cold_history.head(20))
    raise RuntimeError("Cold target cases must have zero strict pre-target and training-safe history.")

stage1_eligible_supply_by_regime = (
    final_sampling_pool_df["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)
MINIMAL_REGIME = stage1_eligible_supply_by_regime.idxmin()
MINIMAL_REGIME_SUPPLY = int(stage1_eligible_supply_by_regime.loc[MINIMAL_REGIME])

if MINIMAL_REGIME_SUPPLY <= 0:
    raise RuntimeError(
        "Minimal regime has no Stage 1 eligible supply available: "
        f"minimal_regime={MINIMAL_REGIME}, "
        f"supply={stage1_eligible_supply_by_regime.to_dict()}"
    )

TARGET_PER_REGIME = int(MINIMAL_REGIME_SUPPLY)
SAMPLE_N_BY_REGIME = {regime: int(TARGET_PER_REGIME) for regime in REGIME_ORDER}
USER_REGIME_QUOTAS = SAMPLE_N_BY_REGIME.copy()
EXPECTED_FINAL_TOTAL_N = int(sum(SAMPLE_N_BY_REGIME.values()))
EXPECTED_UNIQUE_USERS_N = EXPECTED_FINAL_TOTAL_N
TARGET_TOTAL_N = EXPECTED_FINAL_TOTAL_N
TARGET_UNIQUE_USERS = EXPECTED_UNIQUE_USERS_N
EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS = stage1_eligible_supply_by_regime.to_dict()
EXPECTED_ELIGIBLE_POOL_TOTAL_N = int(len(final_sampling_pool_df))
DOWNSTREAM_QUERY_BALANCE_N_PER_REGIME = int(TARGET_PER_REGIME)
DOWNSTREAM_BALANCED_QUERY_REGIME_COUNTS = dict(SAMPLE_N_BY_REGIME)
DOWNSTREAM_BALANCED_QUERY_TOTAL_N = int(EXPECTED_FINAL_TOTAL_N)

short_supply = {
    regime: int(count)
    for regime, count in stage1_eligible_supply_by_regime.items()
    if int(count) < TARGET_PER_REGIME
}
if short_supply:
    raise RuntimeError(
        "Minimal-regime target cannot be met by every regime after candidate-specific "
        "pre-target and Stage 1 history validation: "
        f"target={TARGET_PER_REGIME}, supply={stage1_eligible_supply_by_regime.to_dict()}"
    )

print("Eligible review-case supply before user dedup by regime:")
print(review_case_supply_by_regime.to_string())
print("Strict-last eligible users by regime:")
print(strict_last_supply_by_regime.to_string())
print("Most-recent eligible users by regime:")
print(most_recent_eligible_supply_by_regime.to_string())
print(f"Rank <= {MAX_TARGET_RANK_ALLOWED} eligible users after user dedup by regime:")
print(rank_limited_eligible_supply_by_regime.to_string())
print("Selected user-level pool by strict pre-target regime:")
print(stage1_eligible_supply_by_regime.to_string())
print("Users with no eligible review among top-ranked candidates:", len(users_with_no_eligible_recent_review_df))
print("Selected targets by rank:", selected_target_rank_counts)
print("Minimal regime:", MINIMAL_REGIME)
print("Minimal-regime target per regime:", TARGET_PER_REGIME)
print("Resolved sample counts:", SAMPLE_N_BY_REGIME)
print("Target candidates with prior same-item review:", repeat_target_candidate_n)
print("Users with prior same-item target candidates:", repeat_target_candidate_user_n)
print("Target prior same-item policy:", TARGET_PRIOR_SAME_ITEM_POLICY)


Non-cold selected cases with zero training-safe prior items: 0
Eligible review-case supply before user dedup by regime:
regime
cold        22664
weak         5075
moderate     1278
strong       2942
Strict-last eligible users by regime:
regime
cold        19785
weak         3795
moderate      571
strong        832
Most-recent eligible users by regime:
regime
cold        22369
weak         4153
moderate      698
strong       1128
Rank <= 5 eligible users after user dedup by regime:
regime
cold        22369
weak         4153
moderate      698
strong       1128
Selected user-level pool by strict pre-target regime:
regime
cold        22369
weak         4153
moderate      698
strong       1128
Users with no eligible review among top-ranked candidates: 64064
Selected targets by rank: {1: 24983, 2: 2458, 3: 595, 4: 198, 5: 114}
Minimal regime: moderate
Minimal-regime target per regime: 698
Resolved sample counts: {'cold': 698, 'weak': 698, 'moderate': 698, 'strong': 698}
Target candidates wit

In [12]:
# =========================================================
# User Sampling
# =========================================================
def bucket_numeric_quantiles(series, labels):
    numeric = pd.to_numeric(series, errors="coerce")
    out = pd.Series("unknown", index=series.index, dtype="object")
    valid = numeric.notna()
    if valid.sum() == 0:
        return out
    q = min(len(labels), int(valid.sum()))
    if q <= 1:
        out.loc[valid] = labels[0]
        return out
    ranked = numeric[valid].rank(method="first")
    out.loc[valid] = pd.qcut(ranked, q=q, labels=labels[:q], duplicates="drop").astype(str)
    return out

def normalize_bucket_value(value):
    text = normalize_space(value).lower()
    return text if text else "unknown"

def add_diversity_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["history_length_bucket"] = bucket_numeric_quantiles(out["prior_review_n"], ["low_history", "mid_history", "high_history"])
    out["review_length_bucket"] = bucket_numeric_quantiles(out["target_review_token_count"], ["short_review", "medium_review", "long_review"])
    out["signal_richness_bucket"] = bucket_numeric_quantiles(out["query_safe_signal_total_count"], ["low_signal", "mid_signal", "high_signal"])
    out["recency_bucket"] = bucket_numeric_quantiles(out["review_timestamp_ms"], ["older", "middle", "recent"])
    out["repeat_behavior_bucket"] = np.where(out["prior_item_n"].fillna(0).astype(int) <= 1, "single_or_low_repeat", "repeat_history")
    out["sub_category_bucket"] = out.get("sub_category_norm_text", pd.Series("unknown", index=out.index)).map(normalize_bucket_value)
    out["product_type_bucket"] = out.get("product_type_norm_text", pd.Series("unknown", index=out.index)).map(normalize_bucket_value)
    out["brand_bucket"] = out.get(brand_col_for_sampling, pd.Series("unknown", index=out.index)).map(normalize_bucket_value) if brand_col_for_sampling else "unknown"
    out["diversity_key"] = (
        out["history_length_bucket"].astype(str) + "|" + out["review_length_bucket"].astype(str) + "|" +
        out["signal_richness_bucket"].astype(str) + "|" + out["recency_bucket"].astype(str) + "|" +
        out["repeat_behavior_bucket"].astype(str) + "|" + out["sub_category_bucket"].astype(str) + "|" +
        out["product_type_bucket"].astype(str) + "|" + out["brand_bucket"].astype(str)
    )
    out["sample_priority_score"] = (
        pd.to_numeric(out["query_safe_signal_total_count"], errors="coerce").fillna(0) +
        pd.to_numeric(out["query_safe_signal_family_count"], errors="coerce").fillna(0) +
        np.log1p(pd.to_numeric(out["target_review_token_count"], errors="coerce").fillna(0))
    )
    return out

def deterministic_diversity_sample(df: pd.DataFrame, n: int, random_seed: int) -> pd.DataFrame:
    work = add_diversity_fields(df)
    work["_random_order"] = np.random.default_rng(int(random_seed)).permutation(len(work))
    work = work.sort_values(["diversity_key", "_random_order", "case_id"]).reset_index(drop=True)
    work["_within_diversity_rank"] = work.groupby("diversity_key").cumcount()
    work = work.sort_values(["_within_diversity_rank", "_random_order", "sample_priority_score", "case_id"], ascending=[True, True, False, True]).reset_index(drop=True)
    work["selection_rank_within_regime"] = np.arange(1, len(work) + 1)
    work["selected_for_sampling"] = work["selection_rank_within_regime"] <= int(n)
    work["selection_reason"] = np.where(work["selected_for_sampling"], "diversity_round_robin_within_regime", "not_selected_after_quota")
    return work.drop(columns=["_random_order", "_within_diversity_rank"], errors="ignore")

ranked_parts = []
allocation_rows = []
for regime_index, regime in enumerate(REGIME_ORDER):
    regime_pool = final_sampling_pool_df[final_sampling_pool_df["regime"].eq(regime)].copy()
    ranked = deterministic_diversity_sample(regime_pool, TARGET_PER_REGIME, RANDOM_SEED + regime_index * 1000)
    ranked_parts.append(ranked)
    allocation_rows.append({"regime": regime, "available_eligible_cases": int(len(regime_pool)), "target_cases": TARGET_PER_REGIME, "sampled_cases": int(ranked["selected_for_sampling"].sum())})

ranked_sampling_pool_df = pd.concat(ranked_parts, ignore_index=True)
allocation_df = pd.DataFrame(allocation_rows)
sampled_users_df = ranked_sampling_pool_df[ranked_sampling_pool_df["selected_for_sampling"]].copy()
sampled_users_df = sort_by_regime_bracket(sampled_users_df)
sampled_users_df["sampled_regime_rank"] = sampled_users_df.groupby("regime").cumcount() + 1

print("Sampled rows:", len(sampled_users_df))
print(sampled_users_df["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).to_string())


Sampled rows: 2792
regime
cold        698
weak        698
moderate    698
strong      698


In [13]:
# =========================================================
# Sampling Summary
# =========================================================
prior_history_case_df = sampled_users_df[["case_id", "user_id", "parent_asin", "review_timestamp_ms"]].rename(columns={"parent_asin": "target_parent_asin", "review_timestamp_ms": "target_timestamp_ms"}).drop_duplicates().copy()

eligible_prior_history_case_df = final_sampling_pool_df[
    ["case_id", "user_id", "parent_asin", "review_timestamp_ms"]
].rename(
    columns={
        "parent_asin": "target_parent_asin",
        "review_timestamp_ms": "target_timestamp_ms",
    }
).drop_duplicates().copy()

if eligible_prior_history_case_df["case_id"].duplicated().any():
    raise RuntimeError("Eligible-pool prior-history case keys must be unique.")

if set(eligible_prior_history_case_df["case_id"]) != set(final_sampling_pool_df["case_id"]):
    raise RuntimeError("Eligible-pool prior-history cases must cover the full pre-QCHS reserve pool.")

selected_pool_target_times_df = final_sampling_pool_df[["case_id", "user_id", "review_timestamp_ms"]].rename(columns={"review_timestamp_ms": "target_timestamp_ms"}).drop_duplicates().copy()
selected_later_reviews_after_target_df = selected_pool_target_times_df.merge(
    reviews[["user_id", "review_timestamp_ms"]],
    on="user_id", how="left"
)
selected_later_reviews_after_target_df = selected_later_reviews_after_target_df[
    selected_later_reviews_after_target_df["review_timestamp_ms"].notna() &
    (selected_later_reviews_after_target_df["review_timestamp_ms"] > selected_later_reviews_after_target_df["target_timestamp_ms"])
].copy()
users_with_later_discarded_reviews = int(selected_later_reviews_after_target_df["user_id"].nunique())
discarded_later_review_count = int(len(selected_later_reviews_after_target_df))

sampled_later_reviews_after_target_df = prior_history_case_df.merge(
    reviews[["user_id", "review_timestamp_ms"]],
    on="user_id", how="left"
)
sampled_later_reviews_after_target_df = sampled_later_reviews_after_target_df[
    sampled_later_reviews_after_target_df["review_timestamp_ms"].notna() &
    (sampled_later_reviews_after_target_df["review_timestamp_ms"] > sampled_later_reviews_after_target_df["target_timestamp_ms"])
].copy()
sampled_users_with_later_discarded_reviews = int(sampled_later_reviews_after_target_df["user_id"].nunique())
sampled_later_discarded_review_count = int(len(sampled_later_reviews_after_target_df))

prior_review_history_df = prior_history_case_df.merge(
    reviews[["user_id", "parent_asin", "review_timestamp_ms", "review_datetime"]],
    on="user_id", how="left"
)
prior_review_history_df = prior_review_history_df[
    prior_review_history_df["review_timestamp_ms"].notna()
    & (prior_review_history_df["review_timestamp_ms"] < prior_review_history_df["target_timestamp_ms"])
    & ~prior_review_history_df["parent_asin"].astype(str).eq(prior_review_history_df["target_parent_asin"].astype(str))
].copy()
prior_review_history_df = prior_review_history_df.rename(columns={
    "parent_asin": "prior_item_id",
    "review_timestamp_ms": "prior_timestamp_ms",
    "review_datetime": "prior_review_datetime",
})
prior_review_history_df = prior_review_history_df[["case_id", "user_id", "target_timestamp_ms", "prior_item_id", "prior_timestamp_ms", "prior_review_datetime"]].sort_values(["case_id", "prior_timestamp_ms", "prior_item_id"]).reset_index(drop=True)

eligible_prior_review_history_df = eligible_prior_history_case_df.merge(
    reviews[["user_id", "parent_asin", "review_timestamp_ms", "review_datetime"]],
    on="user_id",
    how="left",
)
eligible_prior_review_history_df = eligible_prior_review_history_df[
    eligible_prior_review_history_df["review_timestamp_ms"].notna()
    & (
        eligible_prior_review_history_df["review_timestamp_ms"]
        < eligible_prior_review_history_df["target_timestamp_ms"]
    )
    & ~eligible_prior_review_history_df["parent_asin"].astype(str).eq(
        eligible_prior_review_history_df["target_parent_asin"].astype(str)
    )
].copy()
eligible_prior_review_history_df = eligible_prior_review_history_df.rename(
    columns={
        "parent_asin": "prior_item_id",
        "review_timestamp_ms": "prior_timestamp_ms",
        "review_datetime": "prior_review_datetime",
    }
)
eligible_prior_review_history_df = eligible_prior_review_history_df[
    [
        "case_id",
        "user_id",
        "target_timestamp_ms",
        "prior_item_id",
        "prior_timestamp_ms",
        "prior_review_datetime",
    ]
].sort_values(
    ["case_id", "prior_timestamp_ms", "prior_item_id"]
).reset_index(drop=True)

train_cutoff_ms = int(TRAIN_REVIEW_CUTOFF_EXCLUSIVE.value // 1_000_000)
training_prior_review_history_df = eligible_prior_review_history_df[
    eligible_prior_review_history_df["prior_timestamp_ms"]
    < np.minimum(
        eligible_prior_review_history_df["target_timestamp_ms"],
        train_cutoff_ms,
    )
].copy()

eligible_pre_target_counts = (
    eligible_prior_review_history_df.groupby("case_id", dropna=False)
    .agg(
        observed_pre_target_interactions=("prior_item_id", "size"),
        observed_pre_target_unique_items=("prior_item_id", "nunique"),
    )
    .reset_index()
)
eligible_train_safe_counts = (
    training_prior_review_history_df.groupby("case_id", dropna=False)
    .agg(
        observed_train_safe_interactions=("prior_item_id", "size"),
        observed_train_safe_unique_items=("prior_item_id", "nunique"),
    )
    .reset_index()
)
eligible_history_audit_df = final_sampling_pool_df[
    [
        "case_id",
        "regime",
        "n_pre_target_interactions",
        "n_pre_target_unique_items",
        "n_train_safe_interactions",
        "n_train_safe_unique_items",
        "stage1_profile_available",
        "stage2_profile_available",
    ]
].merge(
    eligible_pre_target_counts,
    on="case_id",
    how="left",
    validate="one_to_one",
).merge(
    eligible_train_safe_counts,
    on="case_id",
    how="left",
    validate="one_to_one",
)
observed_history_columns = [
    "observed_pre_target_interactions",
    "observed_pre_target_unique_items",
    "observed_train_safe_interactions",
    "observed_train_safe_unique_items",
]
eligible_history_audit_df[observed_history_columns] = (
    eligible_history_audit_df[observed_history_columns].fillna(0).astype(int)
)

eligible_history_mismatch = (
    eligible_history_audit_df["observed_pre_target_interactions"].ne(
        eligible_history_audit_df["n_pre_target_interactions"]
    )
    | eligible_history_audit_df["observed_pre_target_unique_items"].ne(
        eligible_history_audit_df["n_pre_target_unique_items"]
    )
    | eligible_history_audit_df["observed_train_safe_interactions"].ne(
        eligible_history_audit_df["n_train_safe_interactions"]
    )
    | eligible_history_audit_df["observed_train_safe_unique_items"].ne(
        eligible_history_audit_df["n_train_safe_unique_items"]
    )
)
if eligible_history_mismatch.any():
    display(eligible_history_audit_df.loc[eligible_history_mismatch].head(20))
    raise RuntimeError("Candidate-specific history counts do not match exported prior artifacts.")

max_prior_by_case = prior_review_history_df.groupby("case_id")["prior_timestamp_ms"].max().reset_index(name="max_prior_timestamp_ms") if len(prior_review_history_df) else pd.DataFrame(columns=["case_id", "max_prior_timestamp_ms"])
prior_history_leakage_qc_df = prior_history_case_df.merge(max_prior_by_case, on="case_id", how="left")
prior_history_leakage_qc_df["strictly_prior_ok"] = prior_history_leakage_qc_df["max_prior_timestamp_ms"].isna() | (prior_history_leakage_qc_df["max_prior_timestamp_ms"] < prior_history_leakage_qc_df["target_timestamp_ms"])

prior_history_temporal_policy_ok = bool(prior_history_leakage_qc_df["strictly_prior_ok"].all())
if not prior_history_temporal_policy_ok:
    raise RuntimeError("Prior history contains rows at or after the held-out target timestamp.")

sampled_target_items_df = sampled_users_df[["case_id", "parent_asin"]].rename(columns={"parent_asin": "target_parent_asin"}).copy()
prior_target_item_overlap_df = prior_review_history_df.merge(sampled_target_items_df, on="case_id", how="left")
prior_target_item_overlap_df = prior_target_item_overlap_df[
    prior_target_item_overlap_df["prior_item_id"].astype(str).eq(prior_target_item_overlap_df["target_parent_asin"].astype(str))
].copy()
heldout_target_item_excluded_from_prior_history = prior_target_item_overlap_df.empty
if not heldout_target_item_excluded_from_prior_history:
    raise RuntimeError("Held-out target item appears in prior history for one or more sampled cases.")

eligible_target_items_df = eligible_prior_history_case_df[
    ["case_id", "target_parent_asin"]
].copy()
eligible_prior_target_item_overlap_df = eligible_prior_review_history_df.merge(
    eligible_target_items_df,
    on="case_id",
    how="left",
)
eligible_prior_target_item_overlap_df = eligible_prior_target_item_overlap_df[
    eligible_prior_target_item_overlap_df["prior_item_id"].astype(str).eq(
        eligible_prior_target_item_overlap_df["target_parent_asin"].astype(str)
    )
].copy()
eligible_target_item_excluded_from_prior_history = (
    eligible_prior_target_item_overlap_df.empty
)
if not eligible_target_item_excluded_from_prior_history:
    raise RuntimeError(
        "Held-out target item appears in the full eligible-pool prior history."
    )

regime_stage_counts_df = pd.DataFrame({"regime": REGIME_ORDER}).merge(
    review_case_supply_by_regime.rename("all_candidate_query_convertible_review_cases").reset_index(), on="regime", how="left"
).merge(
    strict_last_user_pool_df.groupby("regime").size().reset_index(name="strict_last_review_users"), on="regime", how="left"
).merge(
    strict_last_query_convertible_pool_df.groupby("regime").size().reset_index(name="strict_last_query_convertible_users"), on="regime", how="left"
).merge(
    most_recent_eligible_user_pool_df.groupby("regime").size().reset_index(name="most_recent_eligible_users"), on="regime", how="left"
).merge(
    final_sampling_pool_df.groupby("regime").size().reset_index(name="selected_user_level_pool"), on="regime", how="left"
).merge(
    sampled_users_df.groupby("regime").size().reset_index(name="sampled_cases"), on="regime", how="left"
).fillna(0)
for col in ["all_candidate_query_convertible_review_cases", "strict_last_review_users", "strict_last_query_convertible_users", "most_recent_eligible_users", "selected_user_level_pool", "sampled_cases"]:
    regime_stage_counts_df[col] = regime_stage_counts_df[col].astype(int)

user_regime_counts_df = regime_stage_counts_df.copy()
user_regime_counts_df["requested_cap"] = TARGET_PER_REGIME
user_regime_counts_df["shortfall"] = (user_regime_counts_df["requested_cap"] - user_regime_counts_df["sampled_cases"]).clip(lower=0)

bracket_distribution_by_regime_df = pd.concat([
    heldout_review_pool_df.groupby(["regime", "sampling_bracket"]).size().reset_index(name="cases").assign(stage="all_window_review_candidates"),
    all_candidate_query_convertible_df.groupby(["regime", "sampling_bracket"]).size().reset_index(name="cases").assign(stage="all_candidate_query_convertible"),
    strict_last_query_convertible_pool_df.groupby(["regime", "sampling_bracket"]).size().reset_index(name="cases").assign(stage="strict_last_query_convertible_users"),
    most_recent_eligible_user_pool_df.groupby(["regime", "sampling_bracket"]).size().reset_index(name="cases").assign(stage="most_recent_eligible_users"),
    final_sampling_pool_df.groupby(["regime", "sampling_bracket"]).size().reset_index(name="cases").assign(stage="selected_user_level_pool"),
    sampled_users_df.groupby(["regime", "sampling_bracket"]).size().reset_index(name="cases").assign(stage="sampled"),
], ignore_index=True)
query_convertibility_failure_df = heldout_review_pool_df.groupby(["regime", "query_convertibility_failure_reason"]).size().reset_index(name="count")
review_quality_summary_by_regime_df = final_sampling_pool_df.groupby("regime").agg(users=("user_id", "nunique"), avg_target_review_token_count=("target_review_token_count", "mean"), avg_query_safe_signal_total_count=("query_safe_signal_total_count", "mean")).reset_index()
review_length_distribution_by_regime_df = sampled_users_df.groupby("regime").agg(sampled_cases=("case_id", "count"), min_review_tokens=("target_review_token_count", "min"), median_review_tokens=("target_review_token_count", "median"), max_review_tokens=("target_review_token_count", "max")).reset_index()
signal_richness_distribution_by_regime_df = sampled_users_df.groupby("regime").agg(sampled_cases=("case_id", "count"), min_signal_total=("query_safe_signal_total_count", "min"), median_signal_total=("query_safe_signal_total_count", "median"), max_signal_total=("query_safe_signal_total_count", "max")).reset_index()
evaluation_window_qc_df = pd.DataFrame([
    {"metric": "evaluation_window_months", "value": int(EVALUATION_WINDOW_MONTHS)},
    {"metric": "evaluation_window_start", "value": EVALUATION_WINDOW_START.isoformat()},
    {"metric": "evaluation_window_end", "value": EVALUATION_WINDOW_END.isoformat()},
    {"metric": "evaluation_window_start_inclusive", "value": bool(EVALUATION_WINDOW_START_INCLUSIVE)},
    {"metric": "target_total_n", "value": TARGET_TOTAL_N},
    {"metric": "target_per_regime", "value": TARGET_PER_REGIME},
    {"metric": "total_review_candidates_in_window", "value": int(len(heldout_review_pool_df))},
    {"metric": "unique_users_in_window", "value": int(heldout_review_pool_df["user_id"].nunique())},
    {"metric": "users_with_strict_last_review", "value": int(len(strict_last_user_pool_df))},
    {"metric": "users_with_multiple_candidate_reviews", "value": users_with_multiple_candidate_reviews},
    {"metric": "average_candidate_reviews_per_user", "value": avg_candidate_reviews_per_user},
    {"metric": "max_candidate_reviews_per_user", "value": max_candidate_reviews_per_user},
    {"metric": "target_selection_mode", "value": TARGET_SELECTION_MODE},
    {"metric": "item_match_rate", "value": float(heldout_review_pool_df["item_metadata_exists"].mean()) if len(heldout_review_pool_df) else 0.0},
    {"metric": "discontinued_exclusion_count", "value": int(heldout_review_pool_df["item_is_discontinued"].sum())},
    {"metric": "all_candidate_query_convertible_review_cases", "value": int(len(all_candidate_query_convertible_df))},
    {"metric": "strict_last_review_query_convertible_users", "value": int(len(strict_last_query_convertible_pool_df))},
    {"metric": "most_recent_eligible_users", "value": int(len(most_recent_eligible_user_pool_df))},
    {"metric": "rank_limited_eligible_users", "value": int(len(rank_limited_eligible_user_pool_df))},
    {"metric": "users_with_no_eligible_recent_review_rank_le5", "value": int(len(users_with_no_eligible_recent_review_df))},
    {"metric": "selected_target_rank_counts", "value": json.dumps(selected_target_rank_counts, sort_keys=True)},
    {"metric": "final_selected_user_level_pool", "value": int(len(final_sampling_pool_df))},
    {"metric": "eligible_pool_before_qchs_selection", "value": int(len(final_sampling_pool_df))},
    {"metric": "sampled_total_n", "value": int(len(sampled_users_df))},
    {"metric": "sampled_unique_users", "value": int(sampled_users_df["user_id"].nunique())},
    {"metric": "users_with_later_discarded_reviews_after_selected_target", "value": users_with_later_discarded_reviews},
    {"metric": "discarded_later_reviews_after_selected_target", "value": discarded_later_review_count},
    {"metric": "sampled_users_with_later_discarded_reviews_after_selected_target", "value": sampled_users_with_later_discarded_reviews},
    {"metric": "sampled_discarded_later_reviews_after_selected_target", "value": sampled_later_discarded_review_count},
    {"metric": "prior_history_strictly_prior_ok", "value": prior_history_temporal_policy_ok},
    {"metric": "heldout_target_item_excluded_from_prior_history", "value": heldout_target_item_excluded_from_prior_history},
    {"metric": "training_review_cutoff_exclusive", "value": TRAIN_REVIEW_CUTOFF_EXCLUSIVE.isoformat()},
    {"metric": "training_review_cutoff_equals_evaluation_window_start", "value": bool(TRAIN_REVIEW_CUTOFF_EXCLUSIVE == EVALUATION_WINDOW_START)},
    {"metric": "training_prior_history_case_scope", "value": "full_eligible_pool_before_qchs_selection"},
    {"metric": "training_prior_history_source_cases", "value": int(len(eligible_prior_history_case_df))},
    {"metric": "training_prior_history_rows", "value": int(len(training_prior_review_history_df))},
    {"metric": "training_prior_history_cases", "value": int(training_prior_review_history_df["case_id"].nunique())},
])
print("Prior history rows:", len(prior_review_history_df))
print("Eligible-pool cases before QCHS selection:", len(eligible_prior_history_case_df))
print("Training-period eligible-pool prior history rows:", len(training_prior_review_history_df))
print("Prior history strictly prior OK:", prior_history_temporal_policy_ok)
print("Held-out target item excluded from prior history:", heldout_target_item_excluded_from_prior_history)
print("Eligible-pool target item excluded from prior history:", eligible_target_item_excluded_from_prior_history)
print("Users with later discarded reviews after selected target:", users_with_later_discarded_reviews)
print("Discarded later reviews after selected target:", discarded_later_review_count)
print("Sampled users with later discarded reviews after selected target:", sampled_users_with_later_discarded_reviews)
print("Sampled discarded later reviews after selected target:", sampled_later_discarded_review_count)
print("User-level sampling diagnostics:")
print(evaluation_window_qc_df.to_string(index=False))


Prior history rows: 34794
Eligible-pool cases before QCHS selection: 28348
Training-period eligible-pool prior history rows: 45702
Prior history strictly prior OK: True
Held-out target item excluded from prior history: True
Eligible-pool target item excluded from prior history: True
Users with later discarded reviews after selected target: 3365
Discarded later reviews after selected target: 4698
Sampled users with later discarded reviews after selected target: 441
Sampled discarded later reviews after selected target: 622
User-level sampling diagnostics:
                                                          metric                                                 value
                                        evaluation_window_months                                                     9
                                         evaluation_window_start                      2022-12-12T14:52:26.427000+00:00
                                           evaluation_window_end                  

In [14]:
# =========================================================
# Output Export
# =========================================================
if "target_timestamp_ms" not in sampled_users_df.columns:
    sampled_users_df["target_timestamp_ms"] = pd.to_numeric(sampled_users_df["review_timestamp_ms"], errors="coerce")

if sampled_users_df["target_timestamp_ms"].isna().any():
    raise RuntimeError("target_timestamp_ms must be non-null in sampled user output.")

# Export strict pre-target histories for the full eligible reserve pool used
# in deterministic same-regime replacement.
STRICT_HISTORY_EXPORT_SCOPE_VERSION = "full_eligible_reserve_scope_v2"
strict_prior_history_export_df = (
    eligible_prior_review_history_df
    .sort_values(["case_id", "prior_timestamp_ms", "prior_item_id"], kind="mergesort")
    .reset_index(drop=True)
    .copy()
)

eligible_case_ids = set(final_sampling_pool_df["case_id"].astype(str))
eligible_non_cold_case_ids = set(
    final_sampling_pool_df.loc[
        final_sampling_pool_df["regime"].ne("cold"), "case_id"
    ].astype(str)
)
strict_history_case_ids = set(strict_prior_history_export_df["case_id"].astype(str))
training_history_case_ids = set(training_prior_review_history_df["case_id"].astype(str))

if not strict_history_case_ids.issubset(eligible_case_ids):
    raise RuntimeError("Strict prior-history export contains cases outside the eligible reserve pool.")
if not training_history_case_ids.issubset(eligible_case_ids):
    raise RuntimeError("Training-safe history export contains cases outside the eligible reserve pool.")
if strict_history_case_ids != eligible_non_cold_case_ids:
    missing = sorted(eligible_non_cold_case_ids - strict_history_case_ids)
    extra = sorted(strict_history_case_ids - eligible_non_cold_case_ids)
    raise RuntimeError(
        "Strict prior-history export does not cover exactly the non-cold eligible reserve pool: "
        f"missing={len(missing)}, extra={len(extra)}, "
        f"missing_sample={missing[:10]}, extra_sample={extra[:10]}"
    )
if training_history_case_ids != eligible_non_cold_case_ids:
    missing = sorted(eligible_non_cold_case_ids - training_history_case_ids)
    extra = sorted(training_history_case_ids - eligible_non_cold_case_ids)
    raise RuntimeError(
        "Training-safe history export does not cover exactly the non-cold eligible reserve pool: "
        f"missing={len(missing)}, extra={len(extra)}, "
        f"missing_sample={missing[:10]}, extra_sample={extra[:10]}"
    )
if len(strict_prior_history_export_df) and not strict_prior_history_export_df[
    "prior_timestamp_ms"
].lt(strict_prior_history_export_df["target_timestamp_ms"]).all():
    raise RuntimeError("Strict prior-history export contains target-time or future interactions.")

history_export_case_coverage_qc = {
    "eligible_pool_cases": int(len(eligible_case_ids)),
    "eligible_non_cold_cases": int(len(eligible_non_cold_case_ids)),
    "strict_history_cases": int(len(strict_history_case_ids)),
    "training_history_cases": int(len(training_history_case_ids)),
    "strict_history_matches_non_cold_pool": True,
    "training_history_matches_non_cold_pool": True,
}

canonical_sample_cols = [
    "case_id", "user_id", "parent_asin", "review_timestamp_ms", "target_timestamp_ms", "review_datetime", "target_review_text",
    "prior_review_n", "prior_item_n", "n_pre_target_interactions", "n_pre_target_unique_items",
    "n_train_safe_interactions", "n_train_safe_unique_items", "stage1_profile_available",
    "stage2_profile_available", "regime", "sampling_bracket", "target_rank_desc", "target_selection_mode",
    "target_review_token_count", "query_safe_token_count", "query_safe_signal_family_count", "query_safe_signal_total_count",
    "query_convertible_flag", "query_convertibility_failure_reason",
    "sampled_regime_rank", "selection_rank_within_regime", "selection_reason",
]
optional_sample_cols = ["title", "brand_meta", "sub_category_norm_text", "product_type_norm_text", "form_norm_text", "review_richness_score", "target_review_token_len", "target_review_structure_signal_count", "history_length_bucket", "review_length_bucket", "signal_richness_bucket", "recency_bucket", "repeat_behavior_bucket", "prior_same_item_review_n", "target_item_repeat_prior_flag"]
sample_cols = [col for col in canonical_sample_cols + optional_sample_cols if col in sampled_users_df.columns]
sampled_users_df = sampled_users_df[sample_cols].copy()

if sampled_users_df["case_id"].duplicated().any():
    raise RuntimeError("case_id must remain unique after timestamp pass-through.")
if sampled_users_df["target_timestamp_ms"].isna().any():
    raise RuntimeError("target_timestamp_ms must remain non-null after sample column selection.")

experiment_item_ids = set(sampled_users_df["parent_asin"].astype(str)).union(set(strict_prior_history_export_df["prior_item_id"].dropna().astype(str)))
experiment_item_subset_df = item_schema[item_schema["parent_asin"].astype(str).isin(experiment_item_ids)].copy()
sampled_user_rejections_df = heldout_review_pool_df[heldout_review_pool_df["query_convertible_flag"].ne(1)].copy()
heldout_review_pool_export_df = heldout_review_pool_df.drop(columns=["target_review_text"], errors="ignore")
final_sampling_pool_export_df = final_sampling_pool_df.drop(columns=["target_review_text"], errors="ignore")
sampled_user_rejections_df = sampled_user_rejections_df.drop(columns=["target_review_text"], errors="ignore")

sampled_users_df.to_parquet(SAMPLED_USERS_PARQUET_PATH, index=False)
sampled_users_df.to_parquet(SAMPLED_USERS_COMPAT_PARQUET_PATH, index=False)
sampled_users_df.to_csv(SAMPLED_USERS_CSV_PATH, index=False, encoding="utf-8-sig")
sampled_user_rejections_df.to_parquet(SAMPLED_USER_REJECTIONS_PARQUET_PATH, index=False)
experiment_item_subset_df.to_parquet(SAMPLED_ITEMS_PARQUET_PATH, index=False)
experiment_item_subset_df.to_parquet(SAMPLED_ITEMS_COMPAT_PARQUET_PATH, index=False)
heldout_review_pool_export_df.to_parquet(HELDOUT_REVIEW_POOL_PATH, index=False)
final_sampling_pool_export_df.to_parquet(FINAL_SAMPLING_POOL_PATH, index=False)
strict_prior_history_export_df.to_parquet(USER_PRIOR_REVIEW_HISTORY_PATH, index=False)
training_prior_review_history_df.to_parquet(TRAIN_PRIOR_REVIEW_HISTORY_PATH, index=False)

user_regime_counts_df.to_csv(OUTPUT_DIR / "face_user_regime_counts.csv", index=False, encoding="utf-8-sig")
regime_stage_counts_df.to_csv(OUTPUT_DIR / "face_regime_stage_counts.csv", index=False, encoding="utf-8-sig")
bracket_distribution_by_regime_df.to_csv(OUTPUT_DIR / "face_sampling_bracket_distribution_by_regime.csv", index=False, encoding="utf-8-sig")
query_convertibility_failure_df.to_csv(OUTPUT_DIR / "face_query_convertibility_failure_reasons.csv", index=False, encoding="utf-8-sig")
evaluation_window_qc_df.to_csv(OUTPUT_DIR / "face_evaluation_window_sampling_qc.csv", index=False, encoding="utf-8-sig")
prior_history_leakage_qc_df.to_csv(OUTPUT_DIR / "face_prior_history_leakage_qc.csv", index=False, encoding="utf-8-sig")
review_quality_summary_by_regime_df.to_csv(OUTPUT_DIR / "face_review_quality_summary_by_regime.csv", index=False, encoding="utf-8-sig")
review_length_distribution_by_regime_df.to_csv(OUTPUT_DIR / "face_review_length_distribution_by_regime.csv", index=False, encoding="utf-8-sig")
signal_richness_distribution_by_regime_df.to_csv(OUTPUT_DIR / "face_signal_richness_distribution_by_regime.csv", index=False, encoding="utf-8-sig")

manifest = {
    "stage": STAGE,
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "artifact_schema_version": SAMPLING_ARTIFACT_SCHEMA_VERSION,
    "project_root": str(PROJECT_ROOT),
    "input_paths": {"raw_reviews": str(REVIEWS_PATH), "item_schema": str(ITEM_SCHEMA_PATH)},
    "output_paths": {
        "sampled_users_parquet": str(SAMPLED_USERS_PARQUET_PATH),
        "sampled_users_compat_parquet": str(SAMPLED_USERS_COMPAT_PARQUET_PATH),
        "sampled_users_csv": str(SAMPLED_USERS_CSV_PATH),
        "sampled_user_rejections_parquet": str(SAMPLED_USER_REJECTIONS_PARQUET_PATH),
        "sampled_items_parquet": str(SAMPLED_ITEMS_PARQUET_PATH),
        "sampled_items_compat_parquet": str(SAMPLED_ITEMS_COMPAT_PARQUET_PATH),
        "heldout_review_pool_parquet": str(HELDOUT_REVIEW_POOL_PATH),
        "final_sampling_pool_parquet": str(FINAL_SAMPLING_POOL_PATH),
        "prior_history_parquet": str(USER_PRIOR_REVIEW_HISTORY_PATH),
        "training_prior_history_parquet": str(TRAIN_PRIOR_REVIEW_HISTORY_PATH),
        "sampling_manifest_json": str(SAMPLING_MANIFEST_PATH),
    },
    "evaluation_window_months": int(EVALUATION_WINDOW_MONTHS),
    "evaluation_window_start": EVALUATION_WINDOW_START.isoformat(),
    "evaluation_window_end": EVALUATION_WINDOW_END.isoformat(),
    "evaluation_window_start_inclusive": EVALUATION_WINDOW_START_INCLUSIVE,
    "target_total_n": TARGET_TOTAL_N,
    "target_per_regime": TARGET_PER_REGIME,
    "target_unique_users": TARGET_UNIQUE_USERS,
    "sample_n_by_regime_requested": REQUESTED_SAMPLE_N_BY_REGIME,
    "sample_n_by_regime_resolved": SAMPLE_N_BY_REGIME,
    "eligible_review_case_supply_by_regime_before_user_dedup": review_case_supply_by_regime.to_dict(),
    "eligible_user_supply_by_regime_after_user_dedup": rank_limited_eligible_supply_by_regime.to_dict(),
    "available_eligible_users_by_regime": stage1_eligible_supply_by_regime.to_dict(),
    "limiting_regime": MINIMAL_REGIME,
    "resolved_per_regime_sample_count": int(TARGET_PER_REGIME),
    "selected_target_rank_counts": selected_target_rank_counts,
    "users_with_no_eligible_recent_review_rank_le5": int(len(users_with_no_eligible_recent_review_df)),
    "dynamic_sample_count_policy": "match_minimal_stage1_eligible_regime_supply",
    "regime_history_scope": "strict_pre_target_excluding_target_item",
    "stage1_profile_history_scope": "strict_pre_target_and_before_training_cutoff_excluding_target_item",
    "stage2_profile_history_scope": "strict_pre_target_excluding_target_item",
    "non_cold_stage1_profile_required": True,
    "target_candidate_history_recomputed_per_target": True,
    "max_targets_per_user": MAX_TARGETS_PER_USER,
    "max_target_rank_allowed": MAX_TARGET_RANK_ALLOWED,
    "target_rank_convention": "target_rank_desc is one-based: 1=latest review, 2=second latest, up to MAX_TARGET_RANK_ALLOWED.",
    "target_selection_mode": TARGET_SELECTION_MODE,
    "target_prior_same_item_policy": TARGET_PRIOR_SAME_ITEM_POLICY,
    "repeat_target_candidate_n": int(repeat_target_candidate_n),
    "repeat_target_candidate_user_n": int(repeat_target_candidate_user_n),
    "heldout_target_item_excluded_from_prior_history": bool(heldout_target_item_excluded_from_prior_history),
    "sampling_decision_source": SAMPLING_DECISION_SOURCE,
    "final_window_choice": FINAL_WINDOW_CHOICE,
    "final_sampling_window": FINAL_WINDOW_CHOICE,
    "user_level_sampling": True,
    "regime_balanced": True,
    "regimes": REGIME_ORDER,
    "regime_definition": REGIME_DEFINITION,
    "query_convertibility_thresholds": {
        "MIN_TARGET_REVIEW_TOKENS": MIN_TARGET_REVIEW_TOKENS,
        "MIN_QUERY_SAFE_TOKEN_COUNT": MIN_QUERY_SAFE_TOKEN_COUNT,
        "MIN_QUERY_SAFE_SIGNAL_FAMILIES": MIN_QUERY_SAFE_SIGNAL_FAMILIES,
        "MIN_QUERY_SAFE_SIGNAL_TOTAL": MIN_QUERY_SAFE_SIGNAL_TOTAL,
        "FINAL_MIN_QUERY_TOKENS": FINAL_MIN_QUERY_TOKENS,
        "FINAL_MAX_QUERY_TOKENS": FINAL_MAX_QUERY_TOKENS,
        "FINAL_MIN_QUERY_SAFE_SIGNAL_FAMILIES": FINAL_MIN_QUERY_SAFE_SIGNAL_FAMILIES,
        "FINAL_MIN_QUERY_SAFE_SIGNAL_TOTAL": FINAL_MIN_QUERY_SAFE_SIGNAL_TOTAL,
    },
    "rating_used": False,
    "sentiment_used": False,
    "llm_used": False,
    "target_review_text_intended_use": "synthetic_query_generation_only",
    "target_review_selection_policy": "select_most_recent_query_convertible_target_among_each_users_five_most_recent_evaluation_window_reviews",
    "prior_history_policy": "full eligible reserve-pool strict pre-target history for Stage 2 and same-regime replacement; target parent excluded",
    "training_prior_history_policy": "all eligible-pool same-user interactions with prior_timestamp_ms < min(target_timestamp_ms, training_review_cutoff_exclusive), excluding the held-out target item",
    "training_prior_history_case_scope": "full_eligible_pool_before_qchs_selection",
    "eligible_pool_rows": int(len(final_sampling_pool_df)),
    "eligible_pool_total_n_expected": EXPECTED_ELIGIBLE_POOL_TOTAL_N,
    "eligible_pool_regime_counts": {
        regime: int((final_sampling_pool_df["regime"] == regime).sum())
        for regime in REGIME_ORDER
    },
    "stage1_eligible_regime_counts": EXPECTED_ELIGIBLE_POOL_REGIME_COUNTS,
    "non_cold_stage1_profile_unavailable_rows": int((
        final_sampling_pool_df["regime"].ne("cold")
        & ~final_sampling_pool_df["stage1_profile_available"]
    ).sum()),
    "downstream_query_balance_n_per_regime": DOWNSTREAM_QUERY_BALANCE_N_PER_REGIME,
    "downstream_balanced_query_regime_counts": DOWNSTREAM_BALANCED_QUERY_REGIME_COUNTS,
    "downstream_balanced_query_total_n": DOWNSTREAM_BALANCED_QUERY_TOTAL_N,
    "downstream_query_balancing_basis": DOWNSTREAM_QUERY_BALANCING_BASIS,
    "final_balanced_sample_rows": int(len(sampled_users_df)),
    "final_balanced_sample_regime_counts": {
        regime: int((sampled_users_df["regime"] == regime).sum())
        for regime in REGIME_ORDER
    },
    "strict_prior_history_rows": int(len(strict_prior_history_export_df)),
    "strict_prior_history_source_cases": int(len(strict_history_case_ids)),
    "strict_prior_history_case_scope": "full_eligible_pool_including_same_regime_reserves",
    "strict_history_export_scope_version": STRICT_HISTORY_EXPORT_SCOPE_VERSION,
    "history_export_case_coverage_qc": history_export_case_coverage_qc,
    "common_critical_sampling_contract": COMMON_CRITICAL_SAMPLING_CONTRACT,
    "common_critical_sampling_contract_sha256": COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256,
    "training_prior_history_rows": int(len(training_prior_review_history_df)),
    "training_review_cutoff_exclusive": TRAIN_REVIEW_CUTOFF_EXCLUSIVE.isoformat(),
    "training_prior_history_source_cases": int(len(eligible_prior_history_case_df)),
    "training_prior_history_rows": int(len(training_prior_review_history_df)),
    "training_prior_history_cases_with_rows": int(training_prior_review_history_df["case_id"].nunique()),
    "prior_history_temporal_policy_ok": prior_history_temporal_policy_ok,
    "later_reviews_after_selected_target_discarded": True,
    "prior_review_text_exported": False,
    "prior_brand_or_identifier_columns_exported": False,
    "prior_rating_exported": False,
    "prior_sentiment_exported": False,
    "prior_prompt_or_response_exported": False,
    "item_review_reputation_created_here": False,
    "item_review_reputation_source": "Notebook 02 historical reviews before evaluation window",
}
with open(SAMPLING_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("Saved sampled cases:", SAMPLED_USERS_PARQUET_PATH)
print("Saved prior history:", USER_PRIOR_REVIEW_HISTORY_PATH)
print("Saved training prior history:", TRAIN_PRIOR_REVIEW_HISTORY_PATH)
print("Saved manifest:", SAMPLING_MANIFEST_PATH)


Saved sampled cases: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_user_regime_sample.parquet
Saved prior history: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_user_prior_review_history.parquet
Saved training prior history: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_user_prior_review_history_training.parquet
Saved manifest: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_user_regime_sampling_manifest.json


In [15]:
# =========================================================
# Final Validation
# =========================================================
expected_sample_n_by_regime = {
    regime: int(SAMPLE_N_BY_REGIME[regime])
    for regime in REGIME_ORDER
}

expected_final_total_n = int(sum(expected_sample_n_by_regime.values()))

if expected_final_total_n != EXPECTED_FINAL_TOTAL_N:
    raise RuntimeError(
        f"EXPECTED_FINAL_TOTAL_N is inconsistent with SAMPLE_N_BY_REGIME: "
        f"EXPECTED_FINAL_TOTAL_N={EXPECTED_FINAL_TOTAL_N}, "
        f"sum(SAMPLE_N_BY_REGIME)={expected_final_total_n}."
    )

if EXPECTED_UNIQUE_USERS_N != EXPECTED_FINAL_TOTAL_N:
    raise RuntimeError(
        f"EXPECTED_UNIQUE_USERS_N is inconsistent with EXPECTED_FINAL_TOTAL_N: "
        f"EXPECTED_UNIQUE_USERS_N={EXPECTED_UNIQUE_USERS_N}, "
        f"EXPECTED_FINAL_TOTAL_N={EXPECTED_FINAL_TOTAL_N}."
    )

required_sample_cols = {
    "case_id",
    "user_id",
    "parent_asin",
    "review_timestamp_ms",
    "target_timestamp_ms",
    "review_datetime",
    "target_review_text",
    "prior_review_n",
    "prior_item_n",
    "n_pre_target_interactions",
    "n_pre_target_unique_items",
    "n_train_safe_interactions",
    "n_train_safe_unique_items",
    "stage1_profile_available",
    "stage2_profile_available",
    "regime",
    "sampling_bracket",
    "target_rank_desc",
    "target_selection_mode",
    "target_review_token_count",
    "query_safe_token_count",
    "query_safe_signal_family_count",
    "query_safe_signal_total_count",
    "query_convertible_flag",
    "query_convertibility_failure_reason",
}

missing_sample_cols = sorted(required_sample_cols - set(sampled_users_df.columns))

if missing_sample_cols:
    raise RuntimeError(f"Final sample missing required columns: {missing_sample_cols}")

if len(sampled_users_df) != EXPECTED_FINAL_TOTAL_N:
    raise RuntimeError(
        f"Expected {EXPECTED_FINAL_TOTAL_N} sampled cases, found {len(sampled_users_df)}"
    )

sampled_unique_user_n = int(sampled_users_df["user_id"].nunique())

if sampled_unique_user_n != EXPECTED_UNIQUE_USERS_N:
    raise RuntimeError(
        f"Expected {EXPECTED_UNIQUE_USERS_N} unique users, found {sampled_unique_user_n}"
    )

if sampled_users_df["user_id"].duplicated().any():
    raise RuntimeError("Final sample contains duplicate user_id values.")

if sampled_users_df.groupby("user_id")["case_id"].nunique().max() > MAX_TARGETS_PER_USER:
    raise RuntimeError("Final sample contains more than one target per user.")

regime_counts = (
    sampled_users_df["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)

actual_sample_n_by_regime = {
    regime: int(regime_counts.get(regime, 0))
    for regime in REGIME_ORDER
}

if actual_sample_n_by_regime != expected_sample_n_by_regime:
    raise RuntimeError(
        f"Regime sample counts do not match expected counts. "
        f"Actual: {actual_sample_n_by_regime}. "
        f"Expected: {expected_sample_n_by_regime}."
    )

if any(col in sampled_users_df.columns for col in ["rating", "target_rating", "sentiment"]):
    raise RuntimeError("Rating or sentiment columns must not be present in final sample.")

if not sampled_users_df["query_convertible_flag"].astype(bool).all():
    raise RuntimeError("All sampled cases must be query-convertible.")

if not sampled_users_df["target_selection_mode"].eq(TARGET_SELECTION_MODE).all():
    raise RuntimeError(f"Target selection mode must be {TARGET_SELECTION_MODE} for all sampled rows.")
if MAX_TARGET_RANK_ALLOWED is not None and not sampled_users_df["target_rank_desc"].le(MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"All sampled target_rank_desc values must be <= {MAX_TARGET_RANK_ALLOWED}.")

if "target_item_repeat_prior_flag" in sampled_users_df.columns:
    if sampled_users_df["target_item_repeat_prior_flag"].fillna(0).astype(int).sum() > 0:
        raise RuntimeError("Final sample contains target items previously reviewed by the same user.")

if "prior_same_item_review_n" in sampled_users_df.columns:
    if sampled_users_df["prior_same_item_review_n"].fillna(0).astype(int).sum() > 0:
        raise RuntimeError("Final sample contains targets with prior same-item reviews.")

targets_outside_window = int((
    ~(
        (sampled_users_df["review_datetime"] > EVALUATION_WINDOW_START)
        & (sampled_users_df["review_datetime"] <= EVALUATION_WINDOW_END)
    )
).sum())
if targets_outside_window:
    raise RuntimeError("All sampled target timestamps must be inside the configured evaluation window.")

if prior_review_history_df["prior_timestamp_ms"].notna().any():
    if not (prior_review_history_df["prior_timestamp_ms"] < prior_review_history_df["target_timestamp_ms"]).all():
        raise RuntimeError("Prior history contains reviews at or after target timestamp.")

    target_pairs = sampled_users_df[["case_id", "parent_asin", "review_timestamp_ms"]].rename(
        columns={
            "parent_asin": "prior_item_id",
            "review_timestamp_ms": "prior_timestamp_ms",
        }
    )

    leaked_target_rows = prior_review_history_df.merge(
        target_pairs,
        on=["case_id", "prior_item_id", "prior_timestamp_ms"],
        how="inner",
    )

    if len(leaked_target_rows):
        raise RuntimeError("Target review appears in prior history export.")

    later_review_pairs = (
        sampled_later_reviews_after_target_df[["case_id", "review_timestamp_ms"]]
        .rename(columns={"review_timestamp_ms": "prior_timestamp_ms"})
        .drop_duplicates()
    )

    leaked_later_rows = prior_review_history_df.merge(
        later_review_pairs,
        on=["case_id", "prior_timestamp_ms"],
        how="inner",
    )

    if len(leaked_later_rows):
        raise RuntimeError("Later discarded review appears in prior history export.")

forbidden_prior_history_cols = {
    "prior_review_text",
    "review_text",
    "raw_review_text",
    "target_review_text",
    "review_title",
    "prior_review_title",
    "brand",
    "brand_meta",
    "brand_primary",
    "brand_facet_text",
    "manufacturer",
    "seller",
    "title",
    "item_title",
    "parent_asin",
    "rating",
    "overall",
    "score",
    "sentiment",
    "prompt",
    "response",
    "llm_response",
}

for frame_name, prior_frame in {
    "sampled_prior_history": prior_review_history_df,
    "training_prior_history": training_prior_review_history_df,
}.items():
    forbidden_prior_history_present = sorted(
        forbidden_prior_history_cols & set(prior_frame.columns)
    )
    if forbidden_prior_history_present:
        raise RuntimeError(
            f"{frame_name} contains forbidden text, brand, rating, sentiment, prompt, or response columns: "
            f"{forbidden_prior_history_present}"
        )

eligible_case_ids = set(final_sampling_pool_df["case_id"].astype(str))
eligible_prior_case_ids = set(eligible_prior_history_case_df["case_id"].astype(str))
if eligible_prior_case_ids != eligible_case_ids:
    raise RuntimeError(
        "Eligible prior-history case frame does not cover the full pre-QCHS eligible pool."
    )

training_prior_case_ids = set(
    training_prior_review_history_df["case_id"].astype(str)
)
if not training_prior_case_ids.issubset(eligible_case_ids):
    raise RuntimeError(
        "Training-period prior history contains cases outside the eligible pool."
    )

eligible_target_pairs = eligible_prior_history_case_df[
    ["case_id", "target_parent_asin"]
].rename(columns={"target_parent_asin": "prior_item_id"})

training_target_overlap = training_prior_review_history_df.merge(
    eligible_target_pairs,
    on=["case_id", "prior_item_id"],
    how="inner",
)
if len(training_target_overlap):
    raise RuntimeError(
        "Training-period prior history contains the held-out target item."
    )

if len(training_prior_review_history_df):
    training_cutoff_ms = int(TRAIN_REVIEW_CUTOFF_EXCLUSIVE.value // 1_000_000)
    effective_cutoff = np.minimum(
        training_prior_review_history_df["target_timestamp_ms"].to_numpy(),
        training_cutoff_ms,
    )
    if not (
        training_prior_review_history_df["prior_timestamp_ms"].to_numpy()
        < effective_cutoff
    ).all():
        raise RuntimeError("Training-period prior history violates the effective cutoff.")

    if not (
        training_prior_review_history_df["prior_timestamp_ms"]
        < training_prior_review_history_df["target_timestamp_ms"]
    ).all():
        raise RuntimeError(
            "Training-period prior history contains future or target-time interactions."
        )

sampled_pre_target_counts = (
    prior_review_history_df.groupby("case_id")
    .agg(
        observed_pre_target_interactions=("prior_item_id", "size"),
        observed_pre_target_unique_items=("prior_item_id", "nunique"),
    )
    .reset_index()
)
sampled_training_prior_counts = (
    training_prior_review_history_df.groupby("case_id")
    .agg(
        observed_train_safe_interactions=("prior_item_id", "size"),
        observed_train_safe_unique_items=("prior_item_id", "nunique"),
    )
    .reset_index()
)
sampled_history_audit = sampled_users_df[
    [
        "case_id",
        "regime",
        "n_pre_target_interactions",
        "n_pre_target_unique_items",
        "n_train_safe_interactions",
        "n_train_safe_unique_items",
        "stage1_profile_available",
        "stage2_profile_available",
    ]
].merge(
    sampled_pre_target_counts,
    on="case_id",
    how="left",
    validate="one_to_one",
).merge(
    sampled_training_prior_counts,
    on="case_id",
    how="left",
    validate="one_to_one",
)
observed_columns = [
    "observed_pre_target_interactions",
    "observed_pre_target_unique_items",
    "observed_train_safe_interactions",
    "observed_train_safe_unique_items",
]
sampled_history_audit[observed_columns] = (
    sampled_history_audit[observed_columns].fillna(0).astype(int)
)

history_count_mismatch = (
    sampled_history_audit["observed_pre_target_interactions"].ne(
        sampled_history_audit["n_pre_target_interactions"]
    )
    | sampled_history_audit["observed_pre_target_unique_items"].ne(
        sampled_history_audit["n_pre_target_unique_items"]
    )
    | sampled_history_audit["observed_train_safe_interactions"].ne(
        sampled_history_audit["n_train_safe_interactions"]
    )
    | sampled_history_audit["observed_train_safe_unique_items"].ne(
        sampled_history_audit["n_train_safe_unique_items"]
    )
)
if history_count_mismatch.any():
    display(sampled_history_audit.loc[history_count_mismatch].head(20))
    raise RuntimeError("Sampled pre-target or training-safe history counts do not match their artifacts.")

if not sampled_users_df["prior_review_n"].eq(
    sampled_users_df["n_pre_target_interactions"]
).all():
    raise RuntimeError("prior_review_n must equal n_pre_target_interactions.")
if not sampled_users_df["prior_item_n"].eq(
    sampled_users_df["n_pre_target_unique_items"]
).all():
    raise RuntimeError("prior_item_n must equal n_pre_target_unique_items.")
if not sampled_users_df["regime"].eq(
    sampled_users_df["n_pre_target_interactions"].map(regime_from_prior_count)
).all():
    raise RuntimeError("Sampled regime labels must be recomputed for each selected target timestamp.")

bad_non_cold_training_prior = sampled_history_audit[
    sampled_history_audit["regime"].ne("cold")
    & sampled_history_audit["observed_train_safe_unique_items"].lt(1)
].copy()
print("Non-cold sampled cases with zero training-safe prior items:", len(bad_non_cold_training_prior))
if len(bad_non_cold_training_prior):
    display(bad_non_cold_training_prior[[
        "case_id", "user_id", "regime",
        "observed_pre_target_interactions", "observed_train_safe_interactions",
        "observed_train_safe_unique_items",
    ]].head(20))

bad_non_cold_strict_prior = sampled_history_audit[
    sampled_history_audit["regime"].ne("cold")
    & sampled_history_audit["observed_pre_target_interactions"].lt(1)
].copy()
if len(bad_non_cold_strict_prior):
    display(bad_non_cold_strict_prior.head(20))
    raise RuntimeError("Non-cold sampled cases must have at least one strict prior interaction.")

bad_cold_history = sampled_history_audit[
    sampled_history_audit["regime"].eq("cold")
    & (
        sampled_history_audit["observed_pre_target_interactions"].ne(0)
        | sampled_history_audit["observed_train_safe_interactions"].ne(0)
    )
].copy()
if len(bad_cold_history):
    display(bad_cold_history.head(20))
    raise RuntimeError("Cold sampled cases must have zero strict pre-target and training-safe history.")

# =========================================================
# Temporal target/history validation summary
# =========================================================
training_cutoff_ms = int(TRAIN_REVIEW_CUTOFF_EXCLUSIVE.value // 1_000_000)

strict_prior_rows_at_or_after_target = int(
    prior_review_history_df["prior_timestamp_ms"].ge(prior_review_history_df["target_timestamp_ms"]).sum()
) if len(prior_review_history_df) else 0
training_prior_rows_at_or_after_target = int(
    training_prior_review_history_df["prior_timestamp_ms"].ge(training_prior_review_history_df["target_timestamp_ms"]).sum()
) if len(training_prior_review_history_df) else 0
training_prior_rows_at_or_after_cutoff = int(
    training_prior_review_history_df["prior_timestamp_ms"].ge(training_cutoff_ms).sum()
) if len(training_prior_review_history_df) else 0

sampled_target_item_map = sampled_users_df[["case_id", "parent_asin"]].rename(
    columns={"parent_asin": "target_parent_asin"}
)
strict_prior_target_item_overlap = prior_review_history_df.merge(
    sampled_target_item_map,
    on="case_id",
    how="left",
)
same_target_item_prior_rows = int(
    strict_prior_target_item_overlap["prior_item_id"].astype(str).eq(
        strict_prior_target_item_overlap["target_parent_asin"].astype(str)
    ).sum()
) if len(strict_prior_target_item_overlap) else 0

eligible_target_item_map = final_sampling_pool_df[["case_id", "parent_asin"]].rename(
    columns={"parent_asin": "target_parent_asin"}
)
training_prior_target_item_overlap = training_prior_review_history_df.merge(
    eligible_target_item_map,
    on="case_id",
    how="left",
)
training_same_target_item_prior_rows = int(
    training_prior_target_item_overlap["prior_item_id"].astype(str).eq(
        training_prior_target_item_overlap["target_parent_asin"].astype(str)
    ).sum()
) if len(training_prior_target_item_overlap) else 0
same_target_item_prior_rows_total = same_target_item_prior_rows + training_same_target_item_prior_rows

regime_mismatch_cases = int((
    sampled_users_df["regime"]
    != sampled_users_df["n_pre_target_interactions"].map(regime_from_prior_count)
).sum())
non_cold_zero_strict_prior_cases = int(len(bad_non_cold_strict_prior))
non_cold_zero_training_safe_prior_cases = int(len(bad_non_cold_training_prior))

training_prior_key_cols = ["case_id", "prior_item_id", "prior_timestamp_ms"]
eligible_prior_keys = eligible_prior_review_history_df[training_prior_key_cols].drop_duplicates()
training_prior_keys = training_prior_review_history_df[training_prior_key_cols].drop_duplicates()
if len(training_prior_keys):
    h_train_not_in_h_pre = training_prior_keys.merge(
        eligible_prior_keys,
        on=training_prior_key_cols,
        how="left",
        indicator=True,
    )
    h_train_not_in_h_pre = h_train_not_in_h_pre[h_train_not_in_h_pre["_merge"].ne("both")]
    h_train_not_in_h_pre_rows = int(len(h_train_not_in_h_pre))
else:
    h_train_not_in_h_pre_rows = 0

if strict_prior_rows_at_or_after_target:
    raise RuntimeError("Strict prior history contains rows at or after the target timestamp.")
if training_prior_rows_at_or_after_target:
    raise RuntimeError("Training-safe prior history contains rows at or after the target timestamp.")
if training_prior_rows_at_or_after_cutoff:
    raise RuntimeError("Training-safe prior history contains rows at or after TRAIN_REVIEW_CUTOFF_EXCLUSIVE.")
if same_target_item_prior_rows_total:
    raise RuntimeError("Prior history contains rows for the held-out target parent_asin.")
if regime_mismatch_cases:
    raise RuntimeError("Stored regime does not match regime recomputed from effective prior count.")
if h_train_not_in_h_pre_rows:
    raise RuntimeError("Training-safe history is not a subset of strict pre-target history.")

temporal_validation_summary_df = pd.DataFrame([{
    "evaluation_window_start": EVALUATION_WINDOW_START.isoformat(),
    "evaluation_window_end": EVALUATION_WINDOW_END.isoformat(),
    "selected_target_count": int(len(sampled_users_df)),
    "selected_target_rank_counts": json.dumps(selected_target_rank_counts, sort_keys=True),
    "eligible_supply_before_user_dedup_by_regime": json.dumps(review_case_supply_by_regime.to_dict(), sort_keys=True),
    "eligible_supply_after_user_dedup_by_regime": json.dumps(rank_limited_eligible_supply_by_regime.to_dict(), sort_keys=True),
    "final_sampled_count_by_regime": json.dumps(actual_sample_n_by_regime, sort_keys=True),
    "users_with_no_eligible_recent_review_rank_le5": int(len(users_with_no_eligible_recent_review_df)),
    "limiting_regime": MINIMAL_REGIME,
    "resolved_per_regime_sample_count": int(TARGET_PER_REGIME),
    "targets_outside_window": targets_outside_window,
    "strict_prior_rows_at_or_after_target": strict_prior_rows_at_or_after_target,
    "training_prior_rows_at_or_after_target": training_prior_rows_at_or_after_target,
    "training_prior_rows_at_or_after_cutoff": training_prior_rows_at_or_after_cutoff,
    "same_target_item_prior_rows": same_target_item_prior_rows_total,
    "regime_mismatch_cases": regime_mismatch_cases,
    "non_cold_cases_with_zero_strict_prior": non_cold_zero_strict_prior_cases,
    "non_cold_cases_with_zero_training_safe_prior": non_cold_zero_training_safe_prior_cases,
    "h_train_not_in_h_pre_rows": h_train_not_in_h_pre_rows,
}])
print(temporal_validation_summary_df.to_string(index=False))
print("Temporal target/history validation: passed")

if item_forbidden_cols:
    raise RuntimeError(f"Item schema unsafe columns were present: {item_forbidden_cols}")

if any(col in sampled_users_df.columns for col in ["review_text", "heldout_review_text"]):
    raise RuntimeError("Raw/heldout review text aliases must not be present in final sample.")

print("Final validation PASS.")
print("Expected sample counts:", expected_sample_n_by_regime)
print("Observed sample counts:", actual_sample_n_by_regime)
print("Expected sampled cases:", EXPECTED_FINAL_TOTAL_N)
print("Observed sampled cases:", len(sampled_users_df))
print("Unique sampled users:", sampled_users_df["user_id"].nunique())
print("Target selection mode:", TARGET_SELECTION_MODE)
print("Regime counts:")
print(regime_counts.to_string())
print("Evaluation window:", EVALUATION_WINDOW_START, "< review_datetime <=", EVALUATION_WINDOW_END)
print("Target review text intended use: synthetic_query_generation_only")
print("Item-level review reputation created here: False")


Non-cold sampled cases with zero training-safe prior items: 0
         evaluation_window_start            evaluation_window_end  selected_target_count                           selected_target_rank_counts                     eligible_supply_before_user_dedup_by_regime                     eligible_supply_after_user_dedup_by_regime                              final_sampled_count_by_regime  users_with_no_eligible_recent_review_rank_le5 limiting_regime  resolved_per_regime_sample_count  targets_outside_window  strict_prior_rows_at_or_after_target  training_prior_rows_at_or_after_target  training_prior_rows_at_or_after_cutoff  same_target_item_prior_rows  regime_mismatch_cases  non_cold_cases_with_zero_strict_prior  non_cold_cases_with_zero_training_safe_prior  h_train_not_in_h_pre_rows
2022-12-12T14:52:26.427000+00:00 2023-09-12T14:52:26.427000+00:00                   2792 {"1": 24983, "2": 2458, "3": 595, "4": 198, "5": 114} {"cold": 22664, "moderate": 1278, "strong": 2942, "weak": 5075}

In [16]:
# =========================================================
# Target Case Metadata Export
# =========================================================
from pathlib import Path

import pandas as pd

SAMPLING_DIR = PROJECT_ROOT / "data" / "processed" / "user_sampling"

TARGET_CASE_METADATA_PARQUET = SAMPLING_DIR / "face_target_case_metadata.parquet"

source_sample = final_sampling_pool_df.copy()

if "target_timestamp_ms" not in source_sample.columns:
    source_sample["target_timestamp_ms"] = pd.to_numeric(
        source_sample["review_timestamp_ms"],
        errors="coerce",
    )

source_cols = [
    "case_id",
    "user_id",
    "parent_asin",
    "target_timestamp_ms",
    "review_datetime",
    "target_rank_desc",
    "target_selection_mode",
    "regime",
    "n_pre_target_interactions",
    "n_pre_target_unique_items",
    "n_train_safe_interactions",
    "n_train_safe_unique_items",
    "stage1_profile_available",
    "stage2_profile_available",
]

missing_source_cols = [col for col in source_cols if col not in source_sample.columns]
if missing_source_cols:
    raise RuntimeError(f"Face sample missing required source columns: {missing_source_cols}")

target_case_metadata = (
    source_sample[source_cols]
    .rename(
        columns={
            "parent_asin": "target_parent_asin",
            "review_datetime": "target_review_datetime",
        }
    )
    .copy()
)

required_cols = [
    "case_id",
    "user_id",
    "target_parent_asin",
    "target_timestamp_ms",
    "target_review_datetime",
    "target_rank_desc",
    "target_selection_mode",
    "regime",
    "n_pre_target_interactions",
    "n_pre_target_unique_items",
    "n_train_safe_interactions",
    "n_train_safe_unique_items",
    "stage1_profile_available",
    "stage2_profile_available",
]

missing_cols = [col for col in required_cols if col not in target_case_metadata.columns]
if missing_cols:
    raise RuntimeError(f"target_case_metadata missing required columns: {missing_cols}")

if target_case_metadata["case_id"].duplicated().any():
    raise RuntimeError("target_case_metadata contains duplicated case_id values.")

target_case_metadata["case_id"] = target_case_metadata["case_id"].astype(str)
target_case_metadata["user_id"] = target_case_metadata["user_id"].fillna("").astype(str)
target_case_metadata["target_parent_asin"] = (
    target_case_metadata["target_parent_asin"].fillna("").astype(str)
)
target_case_metadata["target_timestamp_ms"] = pd.to_numeric(
    target_case_metadata["target_timestamp_ms"],
    errors="coerce",
)

if target_case_metadata["user_id"].eq("").any():
    raise RuntimeError("target_case_metadata contains empty user_id values.")
if target_case_metadata["target_parent_asin"].eq("").any():
    raise RuntimeError("target_case_metadata contains empty target item values.")
if (
    target_case_metadata["target_timestamp_ms"].isna().any()
    or target_case_metadata["target_timestamp_ms"].le(0).any()
):
    raise RuntimeError("target_case_metadata contains invalid target_timestamp_ms values.")
if target_case_metadata["target_review_datetime"].isna().any():
    raise RuntimeError("target_case_metadata contains missing target_review_datetime values.")

target_case_metadata["target_timestamp_ms"] = (
    target_case_metadata["target_timestamp_ms"].astype("int64")
)
for col in [
    "n_pre_target_interactions",
    "n_pre_target_unique_items",
    "n_train_safe_interactions",
    "n_train_safe_unique_items",
]:
    target_case_metadata[col] = pd.to_numeric(
        target_case_metadata[col],
        errors="raise",
    ).astype(int)
for col in ["stage1_profile_available", "stage2_profile_available"]:
    target_case_metadata[col] = target_case_metadata[col].astype(bool)

invalid_non_cold = target_case_metadata[
    target_case_metadata["regime"].ne("cold")
    & target_case_metadata["n_train_safe_unique_items"].lt(1)
]
if len(invalid_non_cold):
    raise RuntimeError(
        "Non-cold target metadata rows must have at least one training-safe prior item."
    )

TARGET_CASE_METADATA_PARQUET.parent.mkdir(parents=True, exist_ok=True)
target_case_metadata.to_parquet(TARGET_CASE_METADATA_PARQUET, index=False)

print("Saved target case metadata:", TARGET_CASE_METADATA_PARQUET)
print("Rows:", len(target_case_metadata))
print("Case scope: full Stage 1 eligible user pool before balanced sampling")
print("Unique case_id:", target_case_metadata["case_id"].nunique())
print("Validation: target-specific history metadata passed")

Saved target case metadata: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_target_case_metadata.parquet
Rows: 28348
Case scope: full Stage 1 eligible user pool before balanced sampling
Unique case_id: 28348
Validation: target-specific history metadata passed


In [17]:
# =========================================================
# Common User Sampling Contract Export
# =========================================================
def summarize_common_frame(frame_name, df):
    rows = []
    if df is None:
        return rows
    for common_name, col in COMMON_SAMPLING_COLUMNS.items():
        rows.append({
            "frame_name": frame_name,
            "common_column": common_name,
            "source_column": col,
            "present": bool(col in df.columns),
            "non_null_rows": int(df[col].notna().sum()) if col in df.columns else 0,
            "unique_values": int(df[col].nunique(dropna=True)) if col in df.columns else 0,
        })
    return rows

common_frame_candidates = {
    "sampled_query_cases": globals().get("sampled_query_cases_export", globals().get("sampled_query_cases_df", globals().get("sampled_users_df"))),
    "target_case_metadata": globals().get("target_case_metadata"),
    "prior_history": globals().get("strict_prior_history_export_df", globals().get("prior_history_sampled", globals().get("prior_history_df", globals().get("prior_review_history_df")))),
    "training_prior_history": globals().get("training_prior_review_history_df"),
}

common_column_rows = []
for frame_name, frame in common_frame_candidates.items():
    common_column_rows.extend(summarize_common_frame(frame_name, frame))

common_columns_df = pd.DataFrame(common_column_rows)
if not common_columns_df.empty:
    COMMON_USER_SAMPLING_COLUMNS_PATH.parent.mkdir(parents=True, exist_ok=True)
    common_columns_df.to_csv(COMMON_USER_SAMPLING_COLUMNS_PATH, index=False, encoding="utf-8-sig")

common_contract = dict(COMMON_USER_SAMPLING_OUTPUT_CONTRACT)
common_contract["output_paths"] = {
    "common_user_sampling_contract": str(COMMON_USER_SAMPLING_CONTRACT_PATH),
    "common_user_sampling_columns": str(COMMON_USER_SAMPLING_COLUMNS_PATH),
}
common_contract["frame_row_counts"] = {
    name: int(len(frame))
    for name, frame in common_frame_candidates.items()
    if frame is not None
}
common_contract["training_prior_history_case_scope"] = (
    "full_eligible_pool_before_qchs_selection"
)
common_contract["regime_history_scope"] = "strict_pre_target_excluding_target_item"
common_contract["stage1_profile_history_scope"] = (
    "strict_pre_target_and_before_training_cutoff_excluding_target_item"
)
common_contract["stage2_profile_history_scope"] = "strict_pre_target_excluding_target_item"
common_contract["common_critical_sampling_contract"] = COMMON_CRITICAL_SAMPLING_CONTRACT
common_contract["common_critical_sampling_contract_sha256"] = COMMON_CRITICAL_SAMPLING_CONTRACT_SHA256
common_contract["strict_history_export_scope_version"] = globals().get(
    "STRICT_HISTORY_EXPORT_SCOPE_VERSION", "sample_only_legacy"
)
common_contract["non_cold_stage1_profile_required"] = True

COMMON_USER_SAMPLING_CONTRACT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(COMMON_USER_SAMPLING_CONTRACT_PATH, "w", encoding="utf-8") as f:
    json.dump(common_contract, f, ensure_ascii=False, indent=2)

print("Saved common user sampling contract:", COMMON_USER_SAMPLING_CONTRACT_PATH)
if not common_columns_df.empty:
    print("Saved common user sampling columns:", COMMON_USER_SAMPLING_COLUMNS_PATH)
    display(common_columns_df.head(30))


Saved common user sampling contract: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage0_user_regime_sampling/face_common_user_sampling_contract.json
Saved common user sampling columns: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/stage0_user_regime_sampling/face_common_user_sampling_columns.csv


,frame_name,common_column,source_column,present,non_null_rows,unique_values
0,sampled_query_cases,case_id,case_id,True,2792,2792
1,sampled_query_cases,user_id,user_id,True,2792,2792
2,sampled_query_cases,target_item_id,parent_asin,True,2792,1995
3,sampled_query_cases,target_timestamp_ms,target_timestamp_ms,True,2792,2792
4,sampled_query_cases,target_review_datetime,review_datetime,True,2792,2792
5,sampled_query_cases,regime,regime,True,2792,4
6,sampled_query_cases,prior_count,prior_review_n,True,2792,133
7,sampled_query_cases,target_rank,target_rank_desc,True,2792,5
8,sampled_query_cases,target_selection_mode,target_selection_mode,True,2792,1
9,sampled_query_cases,n_pre_target_interactions,n_pre_target_interactions,True,2792,133
